In [ ]:
import os
import pandas as pd
import numpy as np
import requests
import re
import unicodedata

In [ ]:
# ------------------------------------------------------------------
# Configuración general
# ------------------------------------------------------------------
import os

# Token (App Token) de datos.gov.co: pégalo entre las comillas.
# Si se deja vacío, se usa la variable de entorno DATOS_GOV_TOKEN; sin token la API funciona igual, pero más lenta.
DATOS_GOV_TOKEN = "B4oNr2DWjHOKHsRxQQhJRF136"

# Tabla contractual ETL SECOP (tabla principal)
TABLA_CONTRACTUAL_URL = (
    "https://objectstorage.us-ashburn-1.oraclecloud.com/p/"
    "JzZQGTd7lnlehP8WTnkpHPNKY4S0odQpZro19h5qfj_GG3wMQG9bGRL07QOxLFcv/"
    "n/idf18zap1paf/b/saigc-dl-sandbox/o/analisis_contratacion/gold_exports/"
    "tabla_maestra_completa/tabla_maestra_completa.csv"
)

# Carpetas. Por defecto, la carpeta donde está el notebook (así funciona en cualquier computador
# sin cambiar nada): el caché en "cache_secop" y el Excel de salida junto al notebook.
CARPETA_CACHE = "cache_secop"
CARPETA_SALIDA = "."

TOKEN = DATOS_GOV_TOKEN.strip() or os.getenv("DATOS_GOV_TOKEN")
print("Carpeta de trabajo:", os.path.abspath("."))
print("Token cargado:", bool(TOKEN))


## **1. Importar tabla de documentos contractuales. Datos abiertos.gov.co**

In [ ]:
import time
URL_API = "https://www.datos.gov.co/resource/dmgg-8hin.json"
headers = {"X-App-Token": TOKEN} if TOKEN else {}

def _pedir_bloque(url, params, intentos=6):
    """Pide un bloque a la API; si el servidor falla (500, 502, 503, 504, 429 o corte) espera y reintenta."""
    for i in range(intentos):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=300)
            if r.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"HTTP {r.status_code}")
            r.raise_for_status()
            return r.json()
        except (requests.HTTPError, requests.ConnectionError, requests.Timeout) as e:
            if i == intentos - 1:
                raise
            espera = 15 * (i + 1)
            print(f"   El servidor respondió con error ({e}). Reintento en {espera} s...")
            time.sleep(espera)


def descargar_socrata(url, where=None, bloque=10_000):
    filas, offset = [], 0
    while True:
        params = {"$limit": bloque, "$offset": offset, "$order": ":id"}
        if where:
            params["$where"] = where
        datos = _pedir_bloque(url, params)
        if not datos:
            break
        filas.extend(datos)
        offset += bloque
        print(f"Descargadas {len(filas):,} filas...")
        if len(datos) < bloque:
            break
    return pd.DataFrame(filas)

# Solo ATENEA (filtrado desde la API por NIT)
df = descargar_socrata(URL_API, where="nit_entidad=901508361")

# La URL viene como {"url": "..."} -> dejar solo el texto
df["url_descarga_documento"] = df["url_descarga_documento"].apply(
    lambda x: x.get("url") if isinstance(x, dict) else x
)

# Renombrar a los nombres que ya usa tu código
df = df.rename(columns={
    "n_mero_de_contrato": "Número de Contrato",
    "id_documento": "ID Documento",
    "proceso": "Proceso",
    "nombre_archivo": "Nombre Documento",
    "tamanno_archivo": "Tamaño Documento",
    "extensi_n": "Extensión",
    "descripci_n": "Descripción",
    "fecha_carga": "Fecha Carga",
    "entidad": "Entidad",
    "nit_entidad": "NIT Entidad",
    "url_descarga_documento": "URL Descarga Documento",
})

# Si un bloque no trae alguna columna, se crea vacía
columnas = ["Número de Contrato", "ID Documento", "Proceso", "Nombre Documento",
            "Tamaño Documento", "Extensión", "Descripción", "Fecha Carga",
            "Entidad", "NIT Entidad", "URL Descarga Documento"]
df = df.reindex(columns=columnas)

# Tipos de datos
df["Tamaño Documento"] = pd.to_numeric(df["Tamaño Documento"], errors="coerce").astype("Int64")
df["Fecha Carga"] = pd.to_datetime(df["Fecha Carga"], errors="coerce")

print(df.shape)
df.head(3)

## **2. Imputar Número de Contrato a partir del Proceso**

In [ ]:
# Normalizar: quitar espacios y tratar textos vacíos como NaN
df["Proceso"] = df["Proceso"].astype("string").str.strip()
df["Número de Contrato"] = (
    df["Número de Contrato"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

# (Control) Procesos que tienen más de un número de contrato distinto
conteo = df.groupby("Proceso")["Número de Contrato"].nunique()
ambiguos = conteo[conteo > 1]
if not ambiguos.empty:
    print(f"Atención: {len(ambiguos)} procesos tienen más de un contrato:")
    print(ambiguos)

Los procesos que tienen más de un contrato, deben ser seguramente procesos con muchos contratistas, como un banco de oferentes o una convocatoria. Estos casos no se podrán imputar porque sería una asignación aleatoria.

In [ ]:
vacios_antes = df["Número de Contrato"].isna().sum()

# Solo imputar en procesos con UN único contrato (en los ambiguos no se puede saber cuál es)
n_contratos = df.groupby("Proceso")["Número de Contrato"].transform("nunique")
contrato_por_proceso = df.groupby("Proceso")["Número de Contrato"].transform("first")
contrato_por_proceso = contrato_por_proceso.where(n_contratos == 1)

df["Número de Contrato"] = df["Número de Contrato"].fillna(contrato_por_proceso)

vacios_despues = df["Número de Contrato"].isna().sum()
print(f"Vacíos antes: {vacios_antes} | imputados: {vacios_antes - vacios_despues} | siguen vacíos: {vacios_despues}")

## **3. Quitar duplicados considerando todas las columnas**

In [ ]:
filas_antes = len(df)
df_final = df.drop_duplicates().reset_index(drop=True)
print(f"Duplicados eliminados: {filas_antes - len(df_final)} | filas finales: {len(df_final)}")

## **4. Tabla contractual ETL SECOP (tabla principal)**

La tabla contractual es la tabla principal: una fila por contrato, se deja **tal como viene** y al final (sección 8) se le agregan las columnas documentales.

Los documentos de la API (`df_final`) quedan como **tabla de trabajo** (`docs`): solo los de contratos que están en la tabla contractual, con el `Año` y la `Referencia del contrato` que se usan en las reglas y en los logs. Esta tabla no se exporta.

In [ ]:
# ------------------------------------------------------------------
# 1. Cargar la tabla contractual ETL SECOP (tabla principal, tal como viene)
# ------------------------------------------------------------------
tabla_contractual = pd.read_csv(TABLA_CONTRACTUAL_URL, low_memory=False, dtype={"id_contrato": "string"})

for col in ["id_contrato", "año_inferido", "referencia_del_contrato (contratos_electronicos)"]:
    if col not in tabla_contractual.columns:
        raise KeyError(f"No se encontró la columna '{col}' en la tabla contractual")

llave_contrato = tabla_contractual["id_contrato"].astype("string").str.strip()
print(f"Tabla contractual: {len(tabla_contractual)} contratos "
      f"({llave_contrato.duplicated().sum()} id_contrato repetidos)")

# ------------------------------------------------------------------
# 2. Tabla de trabajo con los documentos (no se exporta)
# ------------------------------------------------------------------
# Solo los documentos de contratos que están en la tabla contractual: los demás no llegarían a la
# salida, así que no vale la pena descargarlos ni leerlos.
df_final["Número de Contrato"] = df_final["Número de Contrato"].astype("string").str.strip()
en_tabla = df_final["Número de Contrato"].isin(set(llave_contrato.dropna()))
print(f"Documentos de ATENEA: {len(df_final)} | de contratos en la tabla contractual: {en_tabla.sum()} | "
      f"sin contrato o fuera de la tabla (se omiten): {(~en_tabla).sum()}")

docs = df_final[en_tabla].copy().reset_index(drop=True)

# Datos del contrato que usan las reglas (Año: regla "3XX.pdf" de 2026 y minutas 2026) y los logs
datos_contrato = (tabla_contractual.assign(_llave=llave_contrato)
                  .drop_duplicates("_llave").set_index("_llave"))
docs["Año"] = docs["Número de Contrato"].map(datos_contrato["año_inferido"])
docs["Referencia del contrato"] = docs["Número de Contrato"].map(
    datos_contrato["referencia_del_contrato (contratos_electronicos)"])


In [ ]:
docs.head(3)

## **5. Clasificación de los documentos**

Sobre la tabla de trabajo `docs` se marcan con "Si"/"No" los documentos de cada tipo. Estas marcas deciden qué documentos se leen (secciones 6 y 7) y qué URLs van a la tabla contractual (sección 8).

+ Identificación de documentos relacionados a CRP (Certificado de Registro Presupuestal) y CDP (Certificado de Disponibilidad Presupuestal)

+ Identificación de documentos relacionados a estudios/documentos previos

+ Identificación de documentos relacionados a minutas

In [ ]:
df["Nombre Documento"].nunique()

In [ ]:
# ------------------------------------------------------------------
# Normalización de "Nombre Documento" (sin crear columnas en docs)
# ------------------------------------------------------------------
def normalizar(texto):
    if pd.isna(texto):
        return ""
    t = unicodedata.normalize("NFKD", str(texto))
    t = "".join(c for c in t if not unicodedata.combining(c))  # quita tildes
    t = t.upper()
    t = re.sub(r"[\u2010-\u2015\u2212]", "-", t)  # guiones raros -> "-"
    t = t.replace("_", " ")                      # guion bajo -> espacio
    t = re.sub(r"\s+", " ", t)                   # espacios múltiples -> uno
    t = re.sub(r"\s*-\s*", "-", t)               # "RP - X" -> "RP-X"
    return t.strip()

nombre_norm = docs["Nombre Documento"].apply(normalizar)

### **1. Columna "CRP/CDP"**

In [ ]:
# ------------------------------------------------------------------
# 1. Columna "CRP/CDP"
# ------------------------------------------------------------------
patron_crp = re.compile(
    r"(?<![A-Z])CRP(?![A-Z])"              # CRP como palabra
    r"|REGISTRO[ -]?PRESUPUESTAL"          # Registro presupuestal
    r"|(?<![A-Z])(?<!ATENEA-)RP[ -]"       # "RP " o "RP-", excepto "ATENEA-RP-"
    r"|RCP|RPS|RP2"                        # contiene RCP, RPS o RP2
    r"|^3-2024"                            # inicia por "3-2024" (cualquier tamaño)
)

tam_doc = pd.to_numeric(docs["Tamaño Documento"], errors="coerce").fillna(0)
anio_contrato = pd.to_numeric(docs["Año"], errors="coerce")

es_crp = (
    nombre_norm.str.contains(patron_crp, regex=True)
    | (nombre_norm == "RP.PDF")                                         # exactamente "RP.pdf"
    | (nombre_norm.str.startswith("3-2026") & (tam_doc > 2_000_000))    # "3-2026..." y > 2 millones
    | (nombre_norm.str.fullmatch(r"3\d{2}\.PDF") & (anio_contrato == 2026))  # "313.pdf", "322.pdf"... solo 2026
)

docs["CRP/CDP"] = es_crp.map({True: "Si", False: "No"})
print(docs["CRP/CDP"].value_counts())

### **2. Columna "Estudios/documentos previos"**

In [ ]:
# ------------------------------------------------------------------
# 2. Columna "Estudios/documentos previos"
# ------------------------------------------------------------------
SEP = r"[ .\-#:°]*"                          # separadores: espacio . - # : °
NUM = r"(?:NUMERO|NUM|NRO|NO|N)"             # No, No., Nro, N°, Nº, Núm, Número...
UNO = r"(?:0*1(?!\d)|UNO(?![A-Z0-9])|I(?![A-Z0-9]))"  # 1, 01, 001, UNO, I (no 10, 11, II)

patron_previos = re.compile(
    r"PREVIO"                                              # contiene "previo"
    r"|(?<!\d)0*1" + SEP + r"CARPETA"                      # "1. CARPETA", "01-CARPETA"
    r"|CARPETA ?0*1(?!\d)"                                 # "CARPETA 1", "CARPETA01"
    r"|^(?:PAQUETE|PAQ|CARPETA)" + SEP                     # INICIA con PAQUETE / PAQ. / CARPETA
    + r"(?:" + NUM + SEP + r")?" + UNO                     #   + (No./#/N°...) opcional + 1
    + r"|^PRIMER[AO]? (?:PAQUETE|CARPETA)"                 # "Primer paquete", "Primera carpeta"
)

es_previo = (
    nombre_norm.str.contains(patron_previos, regex=True)
    | (nombre_norm == "1.ZIP")                             # exactamente "1.zip"
)

docs["Estudios/documentos previos"] = es_previo.map({True: "Si", False: "No"})

In [ ]:
pruebas = [
    # --- Casos originales ---
    "1.zip",
    "Anexo 1.zip",
    "1. CARPETA CONTRATO.rar",
    "1.Carpeta.zip",
    "CARPETA 1.zip",
    "carpeta_1 soportes.rar",
    "CARPETA 10.zip",
    "11. CARPETA.zip",
    "Estudios Previos.pdf",
    "PAQUETE 1.zip",
    "EVIDENCIAS PAQUETE 1_compressed.pdf",

    # --- PAQUETE: variantes ---
    "PAQUETE No. 1.zip",
    "PAQUETE No. 01.zip",
    "PAQUETE No.1.rar",
    "Paquete N° 1.zip",
    "Paquete Nº1.zip",
    "PAQUETE NRO. 001.zip",
    "Paquete Número 1.rar",
    "PAQUETE # 1.zip",
    "Paq. 1.zip",
    "PAQUETE_1.zip",
    "PAQUETE - 1.zip",
    "PAQUETE UNO.zip",
    "Primer paquete.zip",
    "PAQUETE 1 ESTUDIOS PREVIOS Y SOPORTES.rar",
    "CARPETA NO 01.rar",
    "CARPETA # 1.zip",
    "CARPETA #01 soportes.zip",
    "Carpeta N° 1.zip",
    "CARPETA I.zip",
    "Primera carpeta.rar",
    "01. CARPETA.zip",
    "PAQUETE No. 1" ,  "PAQUETE No. 01" , "CARPETA NO 01", "PAQUETE I", "3. CARPETA IDONEIDAD Y EXPERIENCIA.zip",
    "CARPETA I.zip", "Paquete I.zip", "PAQUETE I.zip"
]

for p in pruebas:
    n = normalizar(p)
    r = bool(patron_previos.search(n)) or n == "1.ZIP"
    print(f"{p:45} -> {'Si' if r else 'No'}")

### **3. Columna "Minutas"**

In [ ]:
nombre_norm = docs["Nombre Documento"].apply(normalizar).astype(str)

# Extensión: aplica para TODAS las reglas
ext = docs["Extensión"].astype(str).str.lower().str.strip().str.lstrip(".")
cumple_ext = pd.Series(ext.isin(["pdf", "docx", "doc"]).to_numpy(dtype=bool), index=docs.index)

# Tamaño: aplica SOLO para la regla complementaria
tam = pd.to_numeric(docs["Tamaño Documento"], errors="coerce").astype(float)
cumple_tam = pd.Series(tam.between(200_000, 800_000).fillna(False).to_numpy(dtype=bool), index=docs.index)

# Reglas
tiene_minuta = nombre_norm.str.contains("MINUTA", regex=False)
regla_directa = tiene_minuta | nombre_norm.str.startswith("CONTRATO ATENEA")
regla_atenea = (
    nombre_norm.str.startswith("ATENEA-")
    & ~tiene_minuta
    & ~nombre_norm.str.contains("CRP", regex=False)
    & cumple_tam
)

# Minutas de modificaciones contractuales: no son la minuta del contrato (no traen las obligaciones)
patron_modificacion = re.compile(
    r"MODIF|TERMINACION|ADICION|PRORROGA|CESION|SUSPENSION|REANUDACION|OTRO ?SI|ACLARATORI|LIQUIDACION"
)
es_modificacion = nombre_norm.str.contains(patron_modificacion, regex=True)

es_directa = cumple_ext & regla_directa & ~es_modificacion
es_minuta  = cumple_ext & (regla_directa | regla_atenea) & ~es_modificacion
print(f"Minutas de modificación excluidas: {(cumple_ext & (regla_directa | regla_atenea) & es_modificacion).sum()}")

# Prioridad por contrato
contrato = docs["Número de Contrato"].astype(object)
tiene_contrato = contrato.notna()
n_minutas   = es_minuta.astype(int).groupby(contrato).transform("sum").fillna(0)
hay_directa = es_directa.astype(int).groupby(contrato).transform("max").fillna(0) > 0

quitar = tiene_contrato & es_minuta & ~es_directa & (n_minutas > 1) & hay_directa
es_minuta_final = es_minuta & ~quitar

# ------------------------------------------------------------------
# Si un contrato tiene más de una minuta, dejar solo la más reciente
# ------------------------------------------------------------------
fecha_carga = pd.to_datetime(docs["Fecha Carga"], errors="coerce")
id_doc = pd.to_numeric(docs["ID Documento"], errors="coerce")

candidatas = pd.DataFrame({
    "contrato": docs["Número de Contrato"].astype(object),
    "fecha": fecha_carga,
    "id": id_doc,
})[es_minuta_final & tiene_contrato]

# Más reciente por fecha de carga; si hay empate, el de ID Documento más alto
idx_mas_reciente = (
    candidatas.sort_values(["fecha", "id"], ascending=False, na_position="last")
    .groupby("contrato")
    .head(1)
    .index
)

no_es_mas_reciente = (es_minuta_final & tiene_contrato) & ~docs.index.isin(idx_mas_reciente)
print(f"Minutas descartadas por no ser la más reciente: {no_es_mas_reciente.sum()}")

es_minuta_final = es_minuta_final & ~no_es_mas_reciente

docs["Minutas"] = np.where(es_minuta_final, "Si", "No")
print(docs["Minutas"].value_counts())

In [ ]:
# ------------------------------------------------------------------
# Análisis: minutas por contrato
# ------------------------------------------------------------------
aux = docs.loc[tiene_contrato, ["Número de Contrato", "Nombre Documento"]].assign(
    minuta_antes=es_minuta[tiene_contrato],
    minuta_final=es_minuta_final[tiene_contrato],
)

resumen_minutas = (
    aux.groupby("Número de Contrato")
    .agg(minutas_antes=("minuta_antes", "sum"), minutas_final=("minuta_final", "sum"))
    .reset_index()
)

nombres = (
    aux[aux["minuta_final"]]
    .groupby("Número de Contrato")["Nombre Documento"]
    .agg(lambda s: " | ".join(sorted(set(s))))
)
resumen_minutas["archivos_minuta"] = resumen_minutas["Número de Contrato"].map(nombres).fillna("")

print("\nContratos según cantidad de minutas:")
print(resumen_minutas["minutas_final"].value_counts().sort_index())

revisar = resumen_minutas[resumen_minutas["minutas_final"] > 1]
sin_minuta = resumen_minutas[resumen_minutas["minutas_final"] == 0]
print(f"\nContratos con MÁS de 1 minuta (revisar): {len(revisar)}")
print(f"Contratos SIN minuta: {len(sin_minuta)}")
revisar

### **4. Columna "CDP Exc."**

In [ ]:
# ------------------------------------------------------------------
# 4. Columna "CDP Exc."
# ------------------------------------------------------------------
nombre_norm = docs["Nombre Documento"].apply(normalizar).astype(str)

patron_cdp = re.compile(r"CDP|DISPONIBILIDAD[ -]?PRESUPUESTAL")

docs["CDP Exc."] = (
    nombre_norm.str.contains(patron_cdp, regex=True)
    .map({True: "Si", False: "No"})
)
print(docs["CDP Exc."].value_counts())

### **5. Columna "CuentaCobro1"**

In [ ]:
# ------------------------------------------------------------------
# 5. Columna "CuentaCobro1"
# ------------------------------------------------------------------
patron_cc1 = re.compile(
    r"(?<![A-Z])(?:CUENTA|CTA)"                    # CUENTA o CTA (no cuenta "ACTA")
    r"[ .\-]*(?:(?:DE|COBRO)[ .\-]*)*"             # "DE COBRO" opcional
    r"(?:(?:NUMERO|NUM|NRO|NO|N)[ .°:]*)?"         # No., N., Nro., N°, Nº opcional
    r"[ #]*0*1(?!\d)"                              # #1, #01, 001, 1 (no 10, 11, 012)
    r"|^CC[ -]?0*01(?!\d)"                         # inicia por CC001
)

docs["CuentaCobro1"] = (
    nombre_norm.str.contains(patron_cc1, regex=True)
    .map({True: "Si", False: "No"})
)
print(docs["CuentaCobro1"].value_counts())

In [ ]:
# Pruebas de las reglas de CDP Exc. y CuentaCobro1
pruebas_cdp = ["CDP 456 BO_C8.pdf", "CERTIFICADO-DE-DISPONIBILIDAD-PRESUPUESTAL-CDP--22e.pdf",
               "Disponibilidad presupuestal.pdf", "CRP 1.pdf"]
pruebas_cc1 = ["Cuenta de cobro #01.pdf", "CUENTA COBRO #1.pdf", "cta N.01 marzo.pdf", "Cuenta de cobro 001.pdf",
               "CUENTA DE COBRO N. 001.pdf", "Cuenta_de_cobro_No_1.pdf", "CTA001.pdf", "Cuenta de cobro N° 1.pdf",
               "cuenta de cobro nº 01.pdf", "CC001.pdf", "CC 001.pdf",
               "Cuenta de cobro #2.pdf", "Cuenta de cobro 10.pdf", "ACTA 001.pdf", "CC002.pdf", "CUENTA MARZO 1.pdf"]
for p in pruebas_cdp:
    print(f"CDP Exc.      {p:55} -> {'Si' if patron_cdp.search(normalizar(p)) else 'No'}")
for p in pruebas_cc1:
    print(f"CuentaCobro1  {p:55} -> {'Si' if patron_cc1.search(normalizar(p)) else 'No'}")

### **6. Columna "contratoEjecucion"**

In [ ]:
# ------------------------------------------------------------------
# 6. Columna "contratoEjecucion"
# ------------------------------------------------------------------
nombre_norm = docs["Nombre Documento"].apply(normalizar).astype(str)

patron_ejecucion = re.compile(r"EN[ -]?EJECUCION")   # "En ejecución", "EN_EJECUCION", "En-ejecucion"

docs["contratoEjecucion"] = (
    nombre_norm.str.contains(patron_ejecucion, regex=True)
    .map({True: "Si", False: "No"})
)
print(docs["contratoEjecucion"].value_counts())

In [ ]:
print(docs.shape)
print(docs.columns)

## **6. Obligaciones específicas**

Para cada contrato se arma **una sola lista** de obligaciones específicas, que se agrega a la tabla contractual en la columna `obligaciones_específicas`:

1. **Primero la minuta** (documentos con "Si" en *Minutas*).
2. **Si la minuta no trae la lista** (por ejemplo, contratos de 2026), se usan los **estudios previos**, del más reciente al más antiguo. Si es un comprimido (.zip, .rar, .7z), se busca dentro el documento de *estudios previos*, ignorando *estudio/análisis del sector*, CDP, certificados, etc.
3. Dentro del documento se ubica el subtítulo **"Obligaciones específicas"** y se toma la lista numerada que sigue, hasta el siguiente subtítulo o cláusula. Se ignoran las obligaciones específicas *de la Agencia/ATENEA*.

**Requisitos (una sola vez):**
+ Librerías: `pip install pymupdf python-docx` (ver la primera celda).
+ **7-Zip** instalado (https://www.7-zip.org) para abrir archivos .rar y .7z.

**Tiempos:** la primera ejecución descarga todos los documentos y puede tardar horas. Todo queda guardado en `CARPETA_CACHE`; si se interrumpe, al volver a correr continúa donde iba. Las siguientes ejecuciones solo procesan documentos nuevos.

El detalle de cada documento revisado queda en `log_obligaciones` y se exporta como hoja "Log obligaciones".

In [ ]:
# ------------------------------------------------------------------
# Configuración
# ------------------------------------------------------------------
# Instalar una sola vez (quita el "#" de la siguiente línea, ejecútala y vuelve a ponerlo):
# !pip install pymupdf python-docx

# La carpeta del caché (CARPETA_CACHE) se define en la configuración general, al inicio del notebook.
# Ahí se guardan los archivos descargados y los resultados ya procesados; si el proceso se interrumpe,
# al volver a correrlo continúa donde iba.

MAX_DESCARGAS_SIMULTANEAS = 2          # descargas en paralelo (más de 2-3 hace que SECOP bloquee)
PAUSA_ENTRE_DESCARGAS = 1.0            # segundos de espera antes de cada descarga
MAX_ESTUDIOS_PREVIOS_POR_CONTRATO = 3  # cuántos estudios previos probar si la minuta no sirve
BORRAR_ARCHIVOS_DESCARGADOS = False    # True = borra cada archivo después de leerlo (ahorra espacio)

# Sesión que se presenta como un navegador: sin esto SECOP responde 403 (acceso denegado)
SESION_SECOP = requests.Session()
SESION_SECOP.headers.update({
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36"),
    "Accept": "application/pdf,application/zip,application/octet-stream,*/*;q=0.8",
    "Accept-Language": "es-CO,es;q=0.9,en;q=0.8",
    "Referer": "https://community.secop.gov.co/",
})


In [ ]:
# Prueba rápida: descarga UN documento antes de lanzar el proceso completo
fila_prueba = docs[docs["Minutas"] == "Si"].iloc[0]
r = SESION_SECOP.get(fila_prueba["URL Descarga Documento"], timeout=60)
tipo = r.headers.get("Content-Type", "")
print("Código HTTP:", r.status_code, "| Tipo:", tipo, "| Tamaño:", len(r.content))
if r.status_code == 200 and "html" not in tipo:
    print("OK: SECOP entrega el archivo. Se puede ejecutar el resto.")
else:
    print("SECOP NO entregó el archivo. Inicio de la respuesta:")
    print(r.text[:600])

### **1. Funciones de lectura y extracción**

In [ ]:
# ------------------------------------------------------------------
# 6.1 Funciones para leer documentos y extraer "Obligaciones específicas"
# ------------------------------------------------------------------
import io
import re
import shutil
import tempfile
import subprocess
import unicodedata
import zipfile
from collections import Counter
from pathlib import Path

import pymupdf          # pip install pymupdf
import docx             # pip install python-docx


def _sin_tildes(t):
    t = unicodedata.normalize("NFKD", str(t))
    return "".join(c for c in t if not unicodedata.combining(c)).upper()


# ---------------------------- Lectura de texto ----------------------------

def _quitar_encabezados(paginas):
    """Quita encabezados/pies de página: líneas que se repiten en varias páginas."""
    lineas_pag = [[l.strip() for l in p.splitlines()] for p in paginas]
    n = len(paginas)
    conteo = Counter()
    for lineas in lineas_pag:
        conteo.update({l for l in lineas if l})
    minimo = max(2, int(n * 0.3))
    # Solo líneas con texto (no "1.", "a)", etc., que se repiten por ser numeración)
    repetidas = {l for l, c in conteo.items()
                 if c >= minimo and n >= 2 and len(l) >= 8
                 and (re.search(r"[A-Za-zÁÉÍÓÚáéíóú]{3}", l) or re.fullmatch(r"[\d/\-\. :]+", l))}
    patron_pagina = re.compile(r"^P[aá]gina\s*\d+\s*(de\s*\d+)?$", re.I)
    limpias = []
    for k, lineas in enumerate(lineas_pag):
        con_texto = [i for i, l in enumerate(lineas) if l]
        bordes = set(con_texto[:15] + con_texto[-10:])
        limpias.append("\n".join(
            l for i, l in enumerate(lineas) if l not in repetidas and not patron_pagina.match(l)
            # número de página suelto ("11") al inicio o final de la página
            and not (i in bordes and re.fullmatch(r"\d{1,3}", l) and int(l) in (k, k + 1, k + 2))
        ))
    return "\n".join(limpias)


MAX_PAGINAS_OCR = 90      # páginas máximas a leer con OCR por documento
DPI_OCR_TEXTO = 200       # resolución del OCR para leer texto corrido (más rápido que la de los CRP)
PAGINAS_EXTRA_OCR = 3     # se para cuando la lista no cambia durante estas páginas seguidas


def texto_pdf(datos, con_obligaciones=False):
    """
    Devuelve (texto, escaneado). Si el PDF es escaneado o ilegible, lo lee con OCR (si está disponible).
    con_obligaciones=True: con OCR deja de leer páginas cuando ya encontró la lista de obligaciones
    (más unas páginas extra por si la lista continúa), para no leer documentos de 90 páginas completos.
    """
    doc = pymupdf.open(stream=datos, filetype="pdf")
    paginas = [p.get_text() for p in doc]
    total = sum(len(p.strip()) for p in paginas)
    escaneado = len(paginas) > 0 and (total / len(paginas) < 100 or texto_ilegible("\n".join(paginas)))
    if escaneado and USAR_OCR and RUTA_TESSDATA:
        paginas, hay_titulo, anterior, estables = [], False, None, 0
        for k, pg in enumerate(doc):
            if k >= MAX_PAGINAS_OCR:
                break
            paginas.append(pagina_ocr(pg, dpi=DPI_OCR_TEXTO)[0] or "")
            if con_obligaciones:
                hay_titulo = hay_titulo or _hay_titulo(paginas[-1])
                if hay_titulo:
                    items = extraer_obligaciones(_quitar_encabezados(paginas))
                    estables = estables + 1 if (items and len(items) == len(anterior or [])) else 0
                    anterior = items
                    if estables >= PAGINAS_EXTRA_OCR:   # la lista ya no crece: no hace falta leer más
                        break
        escaneado = sum(len(p.strip()) for p in paginas) < 100
    doc.close()
    return _quitar_encabezados(paginas), escaneado


def _es_lista_word(p):
    if p._p.pPr is not None and p._p.pPr.numPr is not None:
        return True
    nombre_estilo = (p.style.name if p.style is not None else "") or ""
    return "List" in nombre_estilo or "Lista" in nombre_estilo


def texto_docx(datos):
    """Texto de un Word. Las viñetas/numeración automática se marcan con '§ '."""
    d = docx.Document(io.BytesIO(datos))
    lineas = []
    for p in d.paragraphs:
        lineas.append(("§ " if _es_lista_word(p) else "") + p.text)
    for tabla in d.tables:
        for fila in tabla.rows:
            for celda in fila.cells:
                for p in celda.paragraphs:
                    lineas.append(("§ " if _es_lista_word(p) else "") + p.text)
    return "\n".join(lineas), False


def texto_documento(nombre, datos):
    ext = Path(nombre).suffix.lower()
    if ext == ".pdf":
        return texto_pdf(datos, con_obligaciones=True)
    if ext == ".docx":
        return texto_docx(datos)
    raise ValueError(f"formato {ext} no soportado")


# ------------------- Ubicar y separar la lista de obligaciones -------------------

# 1) Títulos específicos: "OBLIGACIONES ESPECÍFICAS ...", "COMPROMISOS ESPECÍFICOS/ESPECÍFICAS ..."
# (tolera errores típicos del OCR: "COMPRPMISOS", "OBLIGACI0NES", "ESPECIFlCOS", palabras pegadas)
_PATRON_TITULO = re.compile(r"(OBLIGACI[OÓ0]N[EÉ]S|COMPR[OÓP0]M[IÍl1]S[O0]S)\s*ESPEC[IÍl1|]F[IÍl1|]C[OA0]S", re.I)
# 2) Si no hay, títulos generales de la contraparte: "COMPROMISOS DE ETB", "COMPROMISOS DEL ASOCIADO"
_PATRON_TITULO_2 = re.compile(r"COMPROMISOS\s+DEL?\s+(?=\S)", re.I)
# 3) Contratos especiales: "OBLIGACIONES DEL COMODANTE", "OBLIGACIONES DEL OPERADOR", ...
_PATRON_TITULO_3 = re.compile(
    r"OBLIGACIONES\s+(?:A\s+CARGO\s+)?(?:DEL|DE\s+LA|POR\s+PARTE\s+DEL?)\s+"
    r"(COMODANTE|COMODATARIO|OPERADOR|ASOCIADO|ALIADO|COOPERANTE|ENTIDAD\s+EJECUTORA|ARRENDADOR)\b", re.I)
# 4) Último recurso: "OBJETIVOS ESPECÍFICOS"
_PATRON_TITULO_4 = re.compile(r"OBJETIVOS\s+ESPEC[IÍ]FICOS", re.I)


def _hay_titulo(texto):
    return any(p.search(texto) for p in (_PATRON_TITULO, _PATRON_TITULO_2, _PATRON_TITULO_3, _PATRON_TITULO_4))
# Títulos que NO son del contratista/aliado (son de ATENEA, de ambas partes o generales)
_EXCLUIR_TITULO = re.compile(r"AGENCIA|ATENEA|CONTRATANTE|LAS PARTES|LOS COOPERANTES|GENERAL|COMUN|CONJUNT|RELACIONAD")
_ENUMERADOR = re.compile(r"^(\(?[A-Za-z]{1,3}[\.\)]|\d+(\.\d+)*[\.\)]?|[ivxIVX]{1,4}[\.\)])$")
# Inicio de otra cláusula (basta con mayúscula inicial: "Cláusula", "PARÁGRAFO", "DÉCIMA")
_CORTE_CLAUSULA = re.compile(r"^(\(?[A-Z0-9]{1,3}[\.\)]\s*)?(CLAUSULA|PARAGRAFO|DECIM|UNDECIM|DUODECIM|VIGESIM)")
# Otros subtítulos (deben estar en MAYÚSCULAS: "OBLIGACIONES DE ...", "C) OBLIGACIONES", "VALOR", "PLAZO")
_CORTE_SUBTITULO = re.compile(r"^(\(?[A-Z0-9]{1,3}[\.\)]\s*)?(OBLIGACIONES|COMPROMISOS|VALOR|PLAZO|FORMA DE PAGO)\b")
# "8.4. Compromisos específicos de la Agencia", "C) Obligaciones de la Agencia": subtítulo numerado en minúscula
_CORTE_NUMERADO = re.compile(r"^(\d+(\.\d+)+\.?|\(?[A-Z][\.\)])\s*(OBLIGACIONES|COMPROMISOS)\b")
_TITULO_MAYUS = re.compile(r"^(\d+(\.\d+)*\.?|[A-Za-z][\.\)])?\s*[A-ZÁÉÍÓÚÑÜ][A-ZÁÉÍÓÚÑÜ ,;:\-–()/]{9,}$")
_TITULO_MAYUS_NUMERADO = re.compile(r"^(\d+(\.\d+)*[\.\)]|[A-Za-z][\.\)])\s*[A-ZÁÉÍÓÚÑÜ][A-ZÁÉÍÓÚÑÜ ,;:\-–()/]{9,}$")
_LETRA_SOLA = re.compile(r"^([a-z]|[ivx]{1,4})[\.\)]\s*$")   # "k." o "iv." solos = subtítulo siguiente


def _es_corte(s):
    n = _sin_tildes(s)
    if s[:1].isupper() and _CORTE_CLAUSULA.match(n):
        return True
    if _CORTE_NUMERADO.match(n):
        return True
    primera = re.sub(r"^\(?[A-Za-z0-9]{1,3}[\.\)]\s*", "", s).split(" ")[0]
    return primera.isupper() and bool(_CORTE_SUBTITULO.match(n))


def _es_titulo_valido(texto, m):
    """
    Es un subtítulo (no una mención dentro de un párrafo) y no es de ATENEA/las partes.
    - En MAYÚSCULAS: basta con que esté al inicio de la línea (o tras algo corto como "B)" o "DÉCIMA. –").
    - En minúscula ("B) Obligaciones específicas del contratista"): debe ir precedido de un
      enumerador (B), j., 8.3.) o ser una línea corta que empieza con el título.
    """
    inicio_linea = texto.rfind("\n", 0, m.start()) + 1
    antes = texto[inicio_linea:m.start()].strip()
    fin_linea = texto.find("\n", m.end())
    fin_linea = fin_linea if fin_linea != -1 else len(texto)
    if m.group(0) == m.group(0).upper():
        if len(antes) > 40:
            return False
    else:
        if not m.group(0)[:1].isupper():
            return False
        if antes and not _ENUMERADOR.match(antes):
            return False
        if not antes and fin_linea - inicio_linea > 120:
            return False
    # De quién son: texto del título hasta ":" o fin de línea (máx. 80 caracteres)
    resto = texto[m.end():fin_linea].split(":")[0][:80]
    return not _EXCLUIR_TITULO.search(_sin_tildes(resto))


def _cortar_seccion(texto, desde, max_lineas=700):
    """
    Toma el texto desde el título hasta el siguiente corte FUERTE: otra cláusula, otro subtítulo de
    obligaciones/compromisos, un subtítulo numerado en mayúsculas ("3. MODALIDAD DE SELECCIÓN"),
    una "Nota" o un "9.4." suelto. Los títulos en mayúsculas SIN numerar (p. ej. "PRODUCTOS
    ESPERADOS" de una tabla dentro de una obligación) no cortan aquí: se recortan después por ítem.
    """
    lineas = texto[desde:].split("\n")[:max_lineas]
    salida = [lineas[0]]                        # resto de la línea del título
    for k, linea in enumerate(lineas[1:], 1):
        s = linea.strip()
        if not s:
            salida.append(linea)
            continue
        if _es_corte(s) or _TITULO_MAYUS_NUMERADO.match(s):
            break
        if re.match(r"^NOTA\b", _sin_tildes(s)):
            break                                   # "Nota 1. ..."
        if _LETRA_SOLA.match(s) or re.fullmatch(r"\d+(\.\d+)+\.?", s):
            # "k." o "9.4." solo en una línea: es el siguiente subtítulo si lo que sigue está en
            # MAYÚSCULAS; si no, es un ítem de la lista ("1.10" con el texto en la línea siguiente)
            siguiente = next((l.strip() for l in lineas[k + 1:] if l.strip()), "")
            if siguiente and (_TITULO_MAYUS.match(siguiente) or _es_corte(siguiente)):
                break
        salida.append(linea)
    return "\n".join(salida)


def _limpiar_item(texto, cortar_en_numeral=False):
    """
    - Corta el ítem en el primer título en MAYÚSCULAS (tabla o sección que quedó pegada).
    - Quita subtítulos intermedios como "Fase 2: Desarrollo" o "Informe de cumplimiento"
      (línea corta, después de una línea en blanco y sin punto final).
    - En listas subnumeradas (1.1, 1.2...), corta en el siguiente numeral principal ("2. Obligaciones de...").
    """
    lineas = texto.split("\n")
    salida = []
    for k, l in enumerate(lineas):
        s = l.strip()
        if k > 0 and s and _TITULO_MAYUS.match(s):
            break
        if k > 0 and cortar_en_numeral and re.match(r"^\d{1,2}([\.\)]|\.\d{1,2}\.?)(\s+\S|$)", s):
            break
        if re.search(r"\(601\)\s*\d{7}|PBX|agenciaatenea\.gov\.co|Carrera 10 No\.? ?28|L[ií]nea 195"
                     r"|PROPORCIONADOS\s*SER[AÁ]N\s*TRATADOS|POL[IÍ]TICA\s*DE\s*TRATAMI?ENTO\s*DE\s*DATOS\s*PERSONALES\s*DE\s*LA\s*$"
                     r"|medio ambiente antes de imprimir|COPIA NO CONTROLADA", s, re.I):
            continue                               # pie de página de la Agencia
        previa = next((x.strip() for x in reversed(lineas[:k]) if x.strip()), "")
        antes_blanco = k > 0 and (not lineas[k - 1].strip() or previa.endswith("."))
        es_subtitulo = (re.match(r"^(FASE|ETAPA|COMPONENTE|LINEA|LÍNEA)\b", s.upper())
                        or (antes_blanco and 0 < len(s) < 45 and not s.endswith((".", ";", ":", ","))
                            and s[:1].isupper()))
        if s and es_subtitulo and salida:
            continue
        salida.append(l)
    texto = re.sub(r"\s+", " ", "\n".join(salida)).strip(" ;")
    return re.sub(r"(?<=[\.;:])\s+\d{1,3}\.?$", "", texto)      # número suelto al final ("... oferta. 11")


def _cortar_en(seccion, inicios):
    items = []
    for i, m in enumerate(inicios):
        fin = inicios[i + 1].start() if i + 1 < len(inicios) else len(seccion)
        items.append(seccion[m.end():fin])
    return items


_NUM_SIMPLE = re.compile(
    r"(?m)(?:^[ \t]*|(?<=[\.:;] ))(\d{1,2})"
    r"(?:\s*[\.\)\-](?=\s|[A-ZÁÉÍÓÚÑ])"       # "1. " "1) " "1.Realizar"
    r"|\s+(?=[a-záéíóúñ]))"                   # "2 apoyar" (número sin punto)
)
_NUM_SUB = re.compile(r"(?m)(?:^[ \t]*|(?<=[\.:;] ))(\d{1,2})\.(\d{1,2})(?:\.(\d{1,2}))?\.?(?=\s|[A-ZÁÉÍÓÚÑ])")


def _secuencia_simple(seccion):
    candidatos = list(_NUM_SIMPLE.finditer(seccion))
    if not candidatos:
        return []
    mejores = []
    # Normalmente empieza en 1; si el título era el ítem "1." (numeración de Word), empieza en 2
    for inicio in sorted({1, int(candidatos[0].group(1))}):
        if inicio > 5:
            continue
        inicios, esperado = [], inicio
        for m in candidatos:
            n = int(m.group(1))
            if n == esperado or (inicios and n == esperado + 1):   # tolera un número faltante
                inicios.append(m)
                esperado = n + 1
        if len(inicios) > len(mejores):
            mejores = inicios
    return mejores


def _secuencia_sub(seccion):
    """
    Listas con numerales: 1.1, 1.2 ... 2.1 ... o con tres niveles (2.1.1, 2.1.2). Se quedan solo
    las "hojas": un numeral que tiene hijos (p. ej. "2.1 Administración de cartera") es un subtítulo.
    """
    entradas, previo = [], None
    for m in _NUM_SUB.finditer(seccion):
        clave = tuple(int(g) for g in m.groups() if g)
        if previo is None:
            if clave[-1] != 1:
                continue
        elif clave <= previo:                      # números que no siguen el orden: se ignoran
            continue
        entradas.append((clave, m))
        previo = clave
    hojas = []
    for k, (clave, m) in enumerate(entradas):
        siguiente = entradas[k + 1][0] if k + 1 < len(entradas) else None
        if siguiente and len(siguiente) > len(clave) and siguiente[:len(clave)] == clave:
            continue                               # tiene hijos: es subtítulo
        hojas.append(m)
    return hojas


_VINETA = re.compile(r"(?m)^[ \t]*[•●▪■◦○➢►▶✓✔❖\uf0b7\uf0a7\uf0d8\uf076\uf0fc\uf0a8\uf0be\uf02d](?=\s)")
_GUION = re.compile(r"(?m)^[ \t]*[-–](?=\s)")


def _separar_items(seccion):
    """
    Separa la lista de obligaciones. Prueba:
    1., 2., 3. ... (o desde 2.)  |  1.1., 1.2. ... (subnumeración)  |  viñetas de Word  |  a), b), c) ...
    """
    items, sub = [], False
    simple = _secuencia_simple(seccion)
    subs = _secuencia_sub(seccion)
    if len(subs) >= 2 and len(subs) > len(simple):          # lista por fases/numerales: 1.1, 1.2, 2.1...
        items, sub = _cortar_en(seccion, subs), True
    elif len(simple) >= 2:
        items = _cortar_en(seccion, simple)
    if not items and "§ " in seccion:                       # numeración automática de Word
        items = [p for p in seccion.split("§ ")[1:]]
    if not items:                                           # viñetas de PDF: •, ▪, ➢, , -...
        for patron in (_VINETA, _GUION):
            vinetas = list(patron.finditer(seccion))
            if len(vinetas) >= 2:
                items = [x.strip() for x in _cortar_en(seccion, vinetas)]
                break
    if not items:                                           # letras a), b), c)
        letras = list(re.finditer(r"(?m)^[ \t]*([a-z])[\.\)](?=\s|[A-ZÁÉÍÓÚÑ])", seccion))
        inicios, esperado = [], "a"
        for m in letras:
            if m.group(1) == esperado:
                inicios.append(m)
                esperado = chr(ord(esperado) + 1)
        if len(inicios) >= 2:
            items = _cortar_en(seccion, inicios)
    items = [_limpiar_item(it, cortar_en_numeral=sub) for it in items]
    return [it for it in items if len(it) > 5]


def extraer_obligaciones(texto):
    """Devuelve la lista de obligaciones específicas encontradas en el texto ([] si no hay)."""
    for patron in (_PATRON_TITULO, _PATRON_TITULO_2, _PATRON_TITULO_3, _PATRON_TITULO_4):
        for m in patron.finditer(texto):
            if not _es_titulo_valido(texto, m):
                continue
            items = _separar_items(_cortar_seccion(texto, m.end()))
            if items:
                return items
    return []


# ---------------------------- Comprimidos ----------------------------

EXT_COMPRIMIDOS = {".zip", ".rar", ".7z"}
EXT_LEGIBLES = {".pdf", ".docx"}
_EXCLUIR_NOMBRE = re.compile(r"SECTOR|MERCADO|IDONEIDAD|CDP|DISPONIBILIDAD|INSUFICIENCIA|INEXISTENCIA|MATRIZ|RIESGO|ANEXO|FORMATO|SARLAFT|HOJA DE VIDA")


def _buscar_7zip():
    for c in [shutil.which("7z"), shutil.which("7za"),
              r"C:\Program Files\7-Zip\7z.exe", r"C:\Program Files (x86)\7-Zip\7z.exe"]:
        if c and Path(c).exists():
            return c
    return None


RUTA_7ZIP = _buscar_7zip()


def _archivos_de_comprimido(nombre, datos, profundidad=0):
    """Devuelve [(nombre, bytes)] de los PDF/Word dentro del comprimido (incluye comprimidos anidados)."""
    ext = Path(nombre).suffix.lower()
    salida = []
    if ext == ".zip":
        with zipfile.ZipFile(io.BytesIO(datos)) as z:
            internos = [(i.filename, z.read(i)) for i in z.infolist()
                        if not i.is_dir() and "__MACOSX" not in i.filename]
    else:
        if RUTA_7ZIP is None:
            raise RuntimeError("7-Zip no está instalado (necesario para .rar y .7z)")
        with tempfile.TemporaryDirectory() as tmp:
            arch = Path(tmp) / f"archivo{ext}"
            arch.write_bytes(datos)
            destino = Path(tmp) / "x"
            subprocess.run(
                [RUTA_7ZIP, "e", str(arch), f"-o{destino}", "-y", "-aou", "-r",
                 "*.pdf", "*.PDF", "*.docx", "*.DOCX", "*.zip", "*.rar", "*.7z"],
                capture_output=True, check=False,
            )
            internos = [(p.name, p.read_bytes()) for p in destino.rglob("*") if p.is_file()] if destino.exists() else []
    for n, b in internos:
        e = Path(n).suffix.lower()
        if e in EXT_LEGIBLES:
            salida.append((n, b))
        elif e in EXT_COMPRIMIDOS and profundidad < 2:
            salida.extend(_archivos_de_comprimido(n, b, profundidad + 1))
    return salida


def _puntaje_estudio_previo(nombre):
    n = _sin_tildes(Path(nombre).name).replace("_", " ").replace("-", " ")
    if re.search(r"SECTOR|MERCADO", n):                      # estudio/análisis del sector
        return -1
    if "PREVIO" not in n and "ESTUDIO" not in n and _EXCLUIR_NOMBRE.search(n):   # CDP, idoneidad...
        return -1
    p = 0
    if "PREVIO" in n or "PREVO" in n:          # "PREVOS" (error de digitación frecuente)
        p += 2
    if "ESTUDIO" in n:
        p += 2
    if "FINAL" in n:
        p += 1
    return p


def obligaciones_de_archivo(nombre, datos):
    """
    Procesa un archivo descargado (PDF, Word o comprimido).
    Devuelve (lista_obligaciones, estado, archivo_usado).
    """
    ext = Path(nombre).suffix.lower()
    if ext in EXT_LEGIBLES:
        candidatos = [(nombre, datos)]
    elif ext in EXT_COMPRIMIDOS:
        internos = _archivos_de_comprimido(nombre, datos)
        if not internos:
            return [], "comprimido sin PDF/Word", None
        con_puntaje = sorted(
            ((_puntaje_estudio_previo(n), len(b), n, b) for n, b in internos),
            key=lambda x: (x[0], x[1]), reverse=True,
        )
        # Primero los que se llaman "estudio(s) previo(s)"; después el resto (sin sector, CDP, etc.)
        candidatos = [(n, b) for p, _, n, b in con_puntaje if p >= 2] + \
                     [(n, b) for p, _, n, b in con_puntaje if 0 <= p < 2]
        if not candidatos:
            return [], "sin estudio previo en el comprimido", None
    elif ext == ".doc":
        return [], "formato .doc no soportado", None
    else:
        return [], f"extensión {ext} no soportada", None

    hubo_escaneado = False
    for n, b in candidatos:
        try:
            texto, escaneado = texto_documento(n, b)
        except Exception:
            continue
        if escaneado:
            hubo_escaneado = True
            continue
        items = extraer_obligaciones(texto)
        if items:
            return items, "ok", Path(n).name
    if hubo_escaneado:
        return [], "PDF escaneado (sin texto)", None
    return [], "no se encontró la sección", None


# ------------------------------------------------------------------
# Reconocimiento de texto (OCR) para PDF escaneados o con texto ilegible
# ------------------------------------------------------------------
# Usa el OCR que trae PyMuPDF; solo necesita los archivos de idioma de Tesseract (español).
# En Windows: instalar Tesseract (https://github.com/UB-Mannheim/tesseract/wiki) marcando "Spanish".

import os
import glob as _glob

USAR_OCR = True
DPI_OCR = 300
# Si Tesseract está instalado en otra carpeta, escribe aquí la ruta de su carpeta "tessdata"
# (la que contiene spa.traineddata), por ejemplo: r"D:\Programas\Tesseract-OCR\tessdata"
RUTA_TESSDATA_MANUAL = ""


def _buscar_tessdata():
    candidatos = [RUTA_TESSDATA_MANUAL, os.environ.get("TESSDATA_PREFIX", ""),
                  r"C:\Program Files\Tesseract-OCR\tessdata",
                  r"C:\Program Files (x86)\Tesseract-OCR\tessdata",
                  os.path.expanduser(r"~\AppData\Local\Programs\Tesseract-OCR\tessdata"),
                  os.path.expanduser(r"~\AppData\Local\Tesseract-OCR\tessdata")]
    for base in [r"C:\Program Files", r"C:\Program Files (x86)", os.path.expanduser(r"~\AppData\Local"),
                 os.path.expanduser(r"~\AppData\Local\Programs"), r"C:\\"]:
        candidatos += _glob.glob(os.path.join(base, "*esseract*", "tessdata"))
    candidatos += _glob.glob("/usr/share/tesseract-ocr/*/tessdata") + ["/usr/share/tessdata", "/opt/homebrew/share/tessdata"]
    for c in candidatos:
        if c and os.path.exists(os.path.join(c, "spa.traineddata")):
            return c
    return None


RUTA_TESSDATA = _buscar_tessdata()
if USAR_OCR and RUTA_TESSDATA is None:
    print("AVISO: no se encontró Tesseract con idioma español. Los PDF escaneados quedarán sin leer.")
else:
    print("OCR activo. Idioma español en:", RUTA_TESSDATA)


def texto_ilegible(texto):
    """True si la página no tiene texto (escaneada) o el texto sale con símbolos raros (fuente dañada)."""
    t = texto.strip()
    if len(t) < 30:
        return True
    palabras = re.findall(r"\b[A-ZÁÉÍÓÚÑ]{4,}\b|\b[a-záéíóúñ]{4,}\b", t)
    return len(palabras) < max(15, len(t) / 60)


def pagina_ocr(pagina, dpi=None):
    """Devuelve (texto, palabras) de la página leídos con OCR, o (None, None) si no hay OCR disponible."""
    if not (USAR_OCR and RUTA_TESSDATA):
        return None, None
    tp = pagina.get_textpage_ocr(language="spa", dpi=dpi or DPI_OCR, full=True, tessdata=RUTA_TESSDATA)
    return pagina.get_text(textpage=tp), pagina.get_text("words", textpage=tp)


_COLS_5 = ["Rubro", "Descripción Rubro", "Concepto Gasto", "Fondo", "Valor CRP"]
_COLS_6 = ["_vigencia"] + _COLS_5


def _fila_desde_celdas(textos):
    """textos: dict columna -> texto de la celda. Devuelve la fila limpia o None."""
    rubro_txt = textos.get("Rubro", "")
    m = re.search(r"[O0o]\s?\d{8,}", rubro_txt)
    if not m:
        return None
    rubro = re.sub(r"\D", "", m.group(0))
    resto = rubro_txt[m.end():]
    for cont in re.findall(r"\b\d{1,5}\b", resto):      # continuación del código en la línea siguiente
        if len(rubro) + len(cont) <= 24:
            rubro += cont
    valor = re.findall(r"\d{1,3}(?:[.,]\d{3})+(?:[.,]\d{2})?", textos.get("Valor CRP", ""))
    vig = re.findall(r"20\d{2}", textos.get("_vigencia", ""))
    def limpio(t):
        t = re.sub(r"\s+", " ", re.sub(r"[|]", " ", t or "")).strip()
        t = re.sub(r"\b0(\d{12,})", r"O\1", t)          # el OCR lee la letra O de los códigos como cero
        return t or None
    return {"Rubro": "O" + rubro[1:], "Descripción Rubro": limpio(textos.get("Descripción Rubro")),
            "Concepto Gasto": limpio(textos.get("Concepto Gasto")), "Fondo": limpio(textos.get("Fondo")),
            "Valor CRP": _a_numero(valor[-1]) if valor else None, "_vigencia": vig[0] if vig else ""}


def _texto_en(rect, palabras):
    ps = [p for p in palabras if rect.contains(pymupdf.Point((p[0] + p[2]) / 2, (p[1] + p[3]) / 2))]
    ps.sort(key=lambda p: (round(p[1] / 4), p[0]))
    return " ".join(p[4] for p in ps)


def ocr_recorte(pagina, rect):
    """OCR de solo un recorte de la página (más confiable para una celda que el OCR de la página completa)."""
    try:
        pix = pagina.get_pixmap(dpi=DPI_OCR, clip=rect)
        pdf = pymupdf.open("pdf", pix.pdfocr_tobytes(language="spa", tessdata=RUTA_TESSDATA))
        return pdf[0].get_text().replace("\n", " ").strip()
    except Exception:
        return ""


def _filas_por_celdas(pagina, palabras):
    """Si la página tiene las líneas de la tabla dibujadas, llena cada celda con las palabras del OCR."""
    filas = []
    for tabla in pagina.find_tables().tables:
        if tabla.col_count not in (5, 6) or len(tabla.rows) < 2:
            continue
        nombres = _COLS_6 if tabla.col_count == 6 else _COLS_5
        encabezado = " ".join(_texto_en(pymupdf.Rect(c), palabras) for c in tabla.rows[0].cells if c)
        if "RUBRO" not in _sin_tildes(encabezado):
            continue
        for fila in tabla.rows[1:]:
            # Cada celda se lee con OCR por separado (más preciso que el OCR de la página completa);
            # si el recorte no da nada, se usa lo que salió en el OCR de la página
            textos = {}
            for n, c in zip(nombres, fila.cells):
                if not c:
                    textos[n] = ""
                    continue
                textos[n] = ocr_recorte(pagina, pymupdf.Rect(c)) or _texto_en(pymupdf.Rect(c), palabras)
            if "TOTAL" in _sin_tildes(" ".join(textos.values())) and not re.search(r"[O0o]\d{8,}", textos.get("Rubro", "")):
                continue
            f = _fila_desde_celdas(textos)
            if f:
                filas.append(f)
    return filas


def _filas_por_coordenadas(palabras, pagina=None):
    """
    Para páginas escaneadas (sin líneas de tabla): arma la tabla RUBRO / DESCRIPCIÓN / CONCEPTO /
    FONDO / VALOR según la posición horizontal de cada palabra bajo los encabezados.
    """
    pal = [(w[0], w[1], w[2], w[3], w[4]) for w in palabras]
    norm = lambda s: _sin_tildes(re.sub(r"[^\wÁÉÍÓÚÑáéíóúñ]", "", s))
    rubros = [p for p in pal if norm(p[4]) == "RUBRO"]
    if not rubros:
        return []
    enc = rubros[0]
    y_enc = (enc[1] + enc[3]) / 2
    en_linea = [p for p in pal if abs((p[1] + p[3]) / 2 - y_enc) < 12]
    columnas, extension, actual = {}, {}, None    # extensión = (x0, x1) del título completo de la columna
    for p in sorted(en_linea, key=lambda p: p[0]):
        n = norm(p[4])
        for clave, nombre in [("RUBRO", "Rubro"), ("DESCRIPCION", "Descripción Rubro"), ("CONCEPTO", "Concepto Gasto"),
                              ("FONDO", "Fondo"), ("VALOR", "Valor CRP"), ("VIGENCIA", "_vigencia")]:
            if n == clave and nombre not in columnas:
                columnas[nombre] = (p[0] + p[2]) / 2
                extension[nombre] = [p[0], p[2]]
                actual = nombre
                break
        else:
            if actual and n:                         # "DEL RUBRO", "DEL GASTO", "FUTURA": parte del mismo título
                extension[actual][1] = p[2]
    if "Valor CRP" not in columnas:
        return []
    orden = sorted(columnas.items(), key=lambda kv: kv[1])
    # El límite entre dos columnas es la mitad del espacio entre sus títulos (no entre sus centros)
    limites = [(extension[orden[i][0]][1] + extension[orden[i + 1][0]][0]) / 2 for i in range(len(orden) - 1)]

    def columna_de(x):
        for i, lim in enumerate(limites):
            if x < lim:
                return orden[i][0]
        return orden[-1][0]

    # Zona de la tabla: desde el encabezado hasta "Total"/"CDP"/"Objeto" o un salto grande entre líneas
    y_ini = max(p[3] for p in en_linea) + 2
    debajo = sorted([p for p in pal if p[1] > y_ini], key=lambda p: p[1])
    y_fin, y_prev = None, y_ini
    for p in debajo:
        if norm(p[4]) in ("TOTAL", "CDP", "OBJETO", "TIPO") or p[1] - y_prev > 22:
            y_fin = p[1] - 1
            break
        y_prev = max(y_prev, p[3])
    y_fin = min(y_fin or y_ini + 200, max(p[3] for p in pal) + 4)     # no pasar de la última palabra leída
    zona = [p for p in pal if y_ini <= (p[1] + p[3]) / 2 < y_fin]
    celdas = [(columna_de((p[0] + p[2]) / 2), p) for p in zona]

    inicios = sorted({round(p[1]) for c, p in celdas
                      if c == "Rubro" and re.fullmatch(r"[O0o]\d{8,}", re.sub(r"[^\w]", "", p[4]))})
    if not inicios:
        return []
    # La primera fila empieza justo bajo el encabezado (la descripción puede empezar más arriba que el rubro)
    bandas = [(min(y_ini, inicios[0] - 6) if i == 0 else inicios[i] - 6, inicios[i + 1] - 6 if i + 1 < len(inicios) else y_fin)
              for i in range(len(inicios))]
    # Límites horizontales de cada columna (para leer cada celda con un OCR propio)
    bordes = [0] + limites + [10000]
    rango_col = {orden[i][0]: (bordes[i], bordes[i + 1]) for i in range(len(orden))}
    filas = []
    for a, b in bandas:
        textos = {}
        for nombre in columnas:
            ps = sorted([p for c, p in celdas if c == nombre and a <= p[1] < b], key=lambda p: (round(p[1] / 4), p[0]))
            textos[nombre] = " ".join(p[4] for p in ps)
            if pagina is not None and nombre != "Rubro":
                x0, x1 = rango_col[nombre]
                recorte = pymupdf.Rect(max(x0, 0), a, min(x1, pagina.rect.width), b) & pagina.rect
                textos[nombre] = ocr_recorte(pagina, recorte) or textos[nombre]
        f = _fila_desde_celdas(textos)
        if f:
            filas.append(f)
    return filas


### **2. Descarga y extracción por contrato**

In [ ]:
# ------------------------------------------------------------------
# 6.2 Descarga (con caché) y extracción por contrato
# ------------------------------------------------------------------
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

carpeta_archivos = Path(CARPETA_CACHE) / "archivos"
carpeta_archivos.mkdir(parents=True, exist_ok=True)
archivo_resultados = Path(CARPETA_CACHE) / "resultados_obligaciones.json"

# Resultados de ejecuciones anteriores (por ID Documento)
resultados = (json.loads(archivo_resultados.read_text(encoding="utf-8"))
              if archivo_resultados.exists() else {})
fallos_descarga = {}   # no se guardan: en la próxima ejecución se reintentan

# Cada vez que se mejoran las reglas de extracción se sube este número: los documentos leídos con
# reglas anteriores se vuelven a leer (desde los archivos ya descargados, sin descargarlos otra vez).
VERSION_REGLAS_OBLIGACIONES = 7
# Desde esta versión, los documentos que YA tienen obligaciones no se vuelven a leer (solo los vacíos)
CONSERVAR_CON_RESULTADO_DESDE = 6


def _hay_que_leer(id_doc):
    r = resultados.get(id_doc, {})
    v = r.get("v", 0)
    if v == VERSION_REGLAS_OBLIGACIONES:
        return False
    return not (r.get("items") and v >= CONSERVAR_CON_RESULTADO_DESDE)


def guardar_resultados():
    archivo_resultados.write_text(json.dumps(resultados, ensure_ascii=False), encoding="utf-8")


def ruta_local(fila):
    ext = str(fila["Extensión"]).lower().strip().lstrip(".")
    if not ext or ext == "nan":
        ext = Path(str(fila["Nombre Documento"])).suffix.lower().lstrip(".")
    return carpeta_archivos / f"{fila['ID Documento']}.{ext}"


_bloqueo = {"seguidos": 0, "avisado": False}
LIMITE_BLOQUEOS = 20   # tras 20 respuestas 403/429 seguidas se dejan de pedir archivos en esta ejecución


FIRMAS = {".pdf": [b"%PDF"], ".zip": [b"PK"], ".rar": [b"Rar!"], ".7z": [b"7z\xbc\xaf"],
          ".docx": [b"PK"], ".doc": [b"\xd0\xcf\x11\xe0"]}


def archivo_valido(ruta_o_bytes, ext):
    """Comprueba que el archivo sea realmente del tipo esperado (y no una página de error de SECOP)."""
    datos = ruta_o_bytes if isinstance(ruta_o_bytes, bytes) else Path(ruta_o_bytes).read_bytes()[:1100]
    firmas = FIRMAS.get(ext.lower())
    if not firmas:
        return len(datos) > 0
    return any(f in datos[:1024] for f in firmas)    # algunos PDF traen bytes antes de "%PDF"


def descargar(fila, intentos=3):
    destino = ruta_local(fila)
    if destino.exists() and destino.stat().st_size > 0:
        if archivo_valido(destino, destino.suffix):
            return True, ""
        destino.unlink()                       # archivo dañado de una ejecución anterior: se descarga de nuevo
    if _bloqueo["seguidos"] >= LIMITE_BLOQUEOS:
        if not _bloqueo["avisado"]:
            _bloqueo["avisado"] = True
            print("   ATENCIÓN: SECOP está bloqueando las descargas (403/429). Se omiten las demás; "
                  "espera un rato (1-2 horas) y vuelve a ejecutar: seguirá donde iba.")
        return False, "omitido: SECOP bloqueando"
    error = ""
    for i in range(intentos):
        try:
            time.sleep(PAUSA_ENTRE_DESCARGAS)
            r = SESION_SECOP.get(fila["URL Descarga Documento"], timeout=180)
            es_html = "text/html" in r.headers.get("Content-Type", "")
            if r.status_code == 200 and r.content and not es_html and not archivo_valido(r.content, destino.suffix):
                error = "SECOP entregó algo que no es el archivo"
                time.sleep(3 * (i + 1))
                continue
            if r.status_code == 200 and r.content and not es_html:
                destino.write_bytes(r.content)
                _bloqueo["seguidos"] = 0
                return True, ""
            error = f"HTTP {r.status_code}" + (" (página web en vez de archivo)" if es_html else "")
            if r.status_code in (403, 429):
                _bloqueo["seguidos"] += 1
                time.sleep(20 * (i + 1))          # espera más larga si SECOP está limitando
                continue
        except Exception as e:
            error = type(e).__name__
        time.sleep(3 * (i + 1))
    return False, error


def procesar(filas, etiqueta):
    """Descarga en paralelo y lee los documentos que aún no tienen resultado."""
    pendientes = [f for _, f in filas.iterrows() if _hay_que_leer(str(f["ID Documento"]))]
    print(f"{etiqueta}: {len(filas)} documentos ({len(filas) - len(pendientes)} ya procesados antes)")
    if not pendientes:
        return

    # 1) Descargar
    with ThreadPoolExecutor(MAX_DESCARGAS_SIMULTANEAS) as ex:
        futuros = {ex.submit(descargar, f): str(f["ID Documento"]) for f in pendientes}
        for k, fut in enumerate(as_completed(futuros), 1):
            ok, error = fut.result()
            if not ok:
                fallos_descarga[futuros[fut]] = error
            if k % 50 == 0 or k == len(pendientes):
                print(f"   descargados {k}/{len(pendientes)}")

    # 2) Leer y extraer
    for k, f in enumerate(pendientes, 1):
        id_doc = str(f["ID Documento"])
        if id_doc in fallos_descarga:
            continue
        ruta = ruta_local(f)
        try:
            items, estado, usado = obligaciones_de_archivo(ruta.name, ruta.read_bytes())
        except Exception as e:
            items, estado, usado = [], f"error al leer ({type(e).__name__}: {e})"[:200], None
        if usado == ruta.name:
            usado = "(el mismo documento)"
        resultados[id_doc] = {"items": items, "estado": estado, "archivo": usado,
                              "v": VERSION_REGLAS_OBLIGACIONES}
        if BORRAR_ARCHIVOS_DESCARGADOS:
            ruta.unlink(missing_ok=True)
        if k % 50 == 0 or k == len(pendientes):
            guardar_resultados()
            print(f"   leídos {k}/{len(pendientes)}")
    guardar_resultados()


def resultado_de(fila):
    id_doc = str(fila["ID Documento"])
    if id_doc in resultados:
        return resultados[id_doc]
    return {"items": [], "estado": f"descarga falló ({fallos_descarga.get(id_doc, '')})", "archivo": None}


con_contrato = docs["Número de Contrato"].notna()
filas_minuta = docs[con_contrato & (docs["Minutas"] == "Si")]
filas_previos = docs[con_contrato & (docs["Estudios/documentos previos"] == "Si")].copy()

obligaciones_contrato = {}   # contrato -> (índice de la fila, lista, fuente)
log = []


def _estado_log(estado, tipo, anio):
    """Desde 2026 las minutas ya no traen las obligaciones específicas (están en los estudios previos)."""
    if tipo == "Minuta" and estado == "no se encontró la sección" and pd.to_numeric(anio, errors="coerce") == 2026:
        return "En 2026 las obligaciones específicas ya no se encuentran en las minutas"
    return estado


def registrar(filas, tipo):
    for idx, f in filas.iterrows():
        r = resultado_de(f)
        contrato = f["Número de Contrato"]
        usado = bool(r["items"]) and contrato not in obligaciones_contrato
        if usado:
            obligaciones_contrato[contrato] = (idx, r["items"], tipo)
        log.append({
            "Número de Contrato": contrato,
            "Referencia del contrato": f.get("Referencia del contrato"),
            "Año": f.get("Año"),
            "ID Documento": f["ID Documento"],
            "Nombre Documento": f["Nombre Documento"],
            "Tipo": tipo,
            "Estado": _estado_log(r["estado"], tipo, f.get("Año")),
            "Archivo leído": r["archivo"],
            "N° obligaciones": len(r["items"]),
            "Usado": "Si" if usado else "No",
            "URL Descarga Documento": f["URL Descarga Documento"],
        })


# Fase 1: minutas (una por contrato)
procesar(filas_minuta, "Fase 1 - Minutas")
registrar(filas_minuta, "Minuta")

# Fase 2: estudios previos, solo para contratos sin resultado. Se prueba del más reciente al más antiguo.
filas_previos["_fecha"] = pd.to_datetime(filas_previos["Fecha Carga"], errors="coerce")
filas_previos["_id"] = pd.to_numeric(filas_previos["ID Documento"], errors="coerce")
filas_previos = filas_previos.sort_values(["_fecha", "_id"], ascending=False)
filas_previos["_orden"] = filas_previos.groupby("Número de Contrato").cumcount()

for ronda in range(MAX_ESTUDIOS_PREVIOS_POR_CONTRATO):
    faltan = filas_previos[
        (filas_previos["_orden"] == ronda)
        & ~filas_previos["Número de Contrato"].isin(list(obligaciones_contrato))
    ]
    if faltan.empty:
        break
    procesar(faltan, f"Fase 2.{ronda + 1} - Estudios previos")
    registrar(faltan, "Estudio previo")

# Texto final por contrato (se agrega a la tabla contractual en la sección 8)
obligaciones_texto = {}
for contrato, (idx, items, tipo) in obligaciones_contrato.items():
    texto = "\n".join(f"{i}. {x}" for i, x in enumerate(items, 1))
    if len(texto) > 32000:                       # límite de una celda de Excel
        texto = texto[:32000] + " …[recortado]"
    obligaciones_texto[contrato] = texto

log_obligaciones = pd.DataFrame(log)

# ------------------------------------------------------------------
# Resumen
# ------------------------------------------------------------------
contratos_candidatos = set(filas_minuta["Número de Contrato"]) | set(filas_previos["Número de Contrato"])
print(f"\nContratos con minuta o estudio previo: {len(contratos_candidatos)}")
print(f"Contratos con obligaciones extraídas:  {len(obligaciones_contrato)}")
print(pd.Series([t for _, _, t in obligaciones_contrato.values()]).value_counts().rename("Fuente").to_string())
if not log_obligaciones.empty:
    print("\nEstado de los documentos leídos:")
    print(log_obligaciones["Estado"].value_counts().to_string())

### **3. Revisión de resultados**

In [ ]:
# ------------------------------------------------------------------
# 6.3 Revisión
# ------------------------------------------------------------------
# Contratos donde no se pudo extraer la lista (para revisar a mano)
if not log_obligaciones.empty:
    sin_obligaciones = (
        log_obligaciones[~log_obligaciones["Número de Contrato"].isin(list(obligaciones_contrato))]
        .groupby("Número de Contrato")
        .agg(documentos_probados=("Nombre Documento", " | ".join), estados=("Estado", " | ".join))
        .reset_index()
    )
    print(f"Contratos sin obligaciones extraídas: {len(sin_obligaciones)}")

# Ejemplo de resultado
for contrato, texto in list(obligaciones_texto.items())[:3]:
    print(f"\n=== {contrato} | {obligaciones_contrato[contrato][2]}")
    print(texto[:800])


## **7. Datos de CRP (registro presupuestal)**

Para cada contrato se extraen: **NumeroCRP, NumeroCDP, Rubro, Descripción Rubro, Concepto Gasto, Fondo, Valor CRP, Fecha CRP y CRP Interno BogData**.

**Orden de búsqueda** (se pasa a la siguiente fuente solo si en la anterior no se encontró nada; si no aparece en ninguna, queda vacío):

1. **CRP directo** (`CRP/CDP` = "Si"): todos los CRP del contrato.
2. **Cuenta de cobro 1** (`CuentaCobro1` = "Si"): se revisan todas las hojas y se toma la del CRP.
3. **CDP** (`CDP Exc.` = "Si"): todos los datos menos NumeroCRP (fecha, valor y código interno son los del CDP).
4. **Contrato en ejecución** (`contratoEjecucion` = "Si"): solo NumeroCDP, NumeroCRP (código del compromiso) y Valor CRP (valor del compromiso; si no aparece, el "valor a utilizar" del CDP).

**Formato del resultado:**
+ Los datos se agregan a la tabla contractual (columnas `numero_crp`, `numero_cdp`, `rubro`, ..., `crp_interno_bogdata`).
+ Si hay varios CRP (o varios rubros), cada columna lleva la lista separada por " | ", del más antiguo al más reciente; la posición 1 de cada columna es el mismo registro. Un dato faltante dentro de una lista se marca con "-".
+ El mismo CRP subido varias veces se cuenta una sola vez.
+ CRP de vigencia futura: `44 (VF 2029)`. Formato del Fondo Cuenta: `FC-456`. Compromisos de vigencia futura en SECOP: `235 (AVF)`.

El detalle de cada documento revisado queda en `log_crp` y se exporta como hoja "Log CRP".

### **1. Funciones de lectura de CRP, CDP y SECOP**

In [ ]:
# ------------------------------------------------------------------
# 7.1 Funciones para leer CRP, CDP y "contrato en ejecución"
# ------------------------------------------------------------------
# Usa pymupdf, re, Path y _archivos_de_comprimido de la sección 6.

MESES = {"ENERO": 1, "FEBRERO": 2, "MARZO": 3, "ABRIL": 4, "MAYO": 5, "JUNIO": 6, "JULIO": 7,
         "AGOSTO": 8, "SEPTIEMBRE": 9, "SETIEMBRE": 9, "OCTUBRE": 10, "NOVIEMBRE": 11, "DICIEMBRE": 12}


def _arreglar_fuente(t):
    """Algunos PDF (formato SEVEN) usan una fuente con letras desplazadas (U+F0xx)."""
    return "".join(chr(ord(c) - 0xF000) if 0xF020 <= ord(c) <= 0xF0FF else c for c in t)


def _a_numero(texto):
    """'87.675.000' -> 87675000 ; '868,051,927.00' -> 868051927 ; '1.200.000.000 COP' -> 1200000000"""
    t = re.sub(r"[^\d.,]", "", str(texto))
    t = re.sub(r"[.,]\d{2}$", "", t)       # quita centavos (",00" o ".00")
    t = re.sub(r"[.,]", "", t)             # quita separadores de miles
    return int(t) if t else None


def _fecha(texto):
    """Busca 'BOGOTÁ D.C. ENERO 29 DE 2026' o '21 de diciembre de 2024' y devuelve 'dd/mm/aaaa'."""
    n = _sin_tildes(texto)
    m = re.search(r"BOGOTA D\.?C\.?\s+([A-Z]+)\s+(\d{1,2})\s+DE\s+(\d{4})", n)
    if m and m.group(1) in MESES:
        return f"{int(m.group(2)):02d}/{MESES[m.group(1)]:02d}/{m.group(3)}"
    for patron in (r"BOGOTA D\.?C\.?\s*(\d{1,2})\s+DE\s+([A-Z]+)\s+DE\s+(\d{4})",
                   r"A LOS\s+(\d{1,2})\s+DE\s+([A-Z]+)\s+DE\s+(\d{4})"):        # "Bogota D.C, a los 6 de mayo de 2024"
        m = re.search(patron, n)
        if m and m.group(2) in MESES:
            return f"{int(m.group(1)):02d}/{MESES[m.group(2)]:02d}/{m.group(3)}"
    return None


def _limpiar_celda(c):
    return re.sub(r"\s+", " ", c or "").strip()


def _filas_desde_texto(texto):
    """
    Respaldo cuando no se detecta la tabla: lee la fila (un solo rubro) directamente del texto,
    entre el encabezado VALOR y la línea Total.
    """
    lineas = [l.strip() for l in texto.splitlines()]
    try:
        i = next(k for k, l in enumerate(lineas) if l == "VALOR")
        j = next(k for k in range(i + 1, len(lineas)) if lineas[k].upper() == "TOTAL")
    except StopIteration:
        return []
    bloque = [l for l in lineas[i + 1:j] if l]
    vig = ""
    if bloque and re.fullmatch(r"20\d{2}", bloque[0]):
        vig, bloque = bloque[0], bloque[1:]
    if not bloque or not re.fullmatch(r"[A-Z]?\d{8,}", bloque[0]):
        return []
    rubro, k = bloque[0], 1
    while k < len(bloque) and re.fullmatch(r"\d{1,6}", bloque[k]):
        rubro += bloque[k]
        k += 1
    valores = [n for n, l in enumerate(bloque) if re.fullmatch(r"[\d.]+(,\d{2})?", l) and n >= k]
    if not valores:
        return []
    v = valores[-1]
    resto = bloque[k:v]
    i_conc = next((n for n, l in enumerate(resto) if re.match(r"[A-Z]\d{10,}\s", l)), None)
    i_fondo = next((n for n, l in enumerate(resto) if re.match(r"\d-\d{3}-[A-Z]\d{3}", l)), None)
    descr = " ".join(resto[:i_conc]) if i_conc is not None else None
    conc = " ".join(resto[i_conc:i_fondo]) if i_conc is not None else None
    fondo = " ".join(resto[i_fondo:]) if i_fondo is not None else None
    return [{"Rubro": rubro, "Descripción Rubro": descr, "Concepto Gasto": conc, "Fondo": fondo,
             "Valor CRP": _a_numero(bloque[v]), "_vigencia": vig}]


def _palabras_recorte(pagina, rect):
    """OCR solo de un recorte de la página; devuelve las palabras con coordenadas de la página."""
    try:
        pix = pagina.get_pixmap(dpi=DPI_OCR, clip=rect)
        pdf = pymupdf.open("pdf", pix.pdfocr_tobytes(language="spa", tessdata=RUTA_TESSDATA))
        escala = rect.width / pdf[0].rect.width
        return [(rect.x0 + w[0] * escala, rect.y0 + w[1] * escala, rect.x0 + w[2] * escala,
                 rect.y0 + w[3] * escala, w[4]) + tuple(w[5:]) for w in pdf[0].get_text("words")]
    except Exception:
        return []


def _franja(pagina, palabras, inicio, fin, alto_max=350):
    """
    Recorte horizontal de la página entre la palabra que cumple 'inicio' y la primera que cumple 'fin'
    más abajo (p. ej. entre '... así:' y 'CDP'). Devuelve un Rect o None.
    """
    norm = lambda s: _sin_tildes(re.sub(r"[^\wÁÉÍÓÚÑáéíóúñ]", "", s))
    arriba = [w for w in palabras if inicio(norm(w[4]), w[4])]
    if not arriba:
        return None
    y0 = arriba[0][3]
    abajo = [w[1] for w in palabras if w[1] > y0 + 5 and fin(norm(w[4]), w[4])]
    y1 = min(abajo) if abajo else y0 + alto_max
    return pymupdf.Rect(pagina.rect.x0, y0, pagina.rect.x1, min(y1, y0 + alto_max))


def _filas_por_franja(pagina, palabras):
    """
    Escaneados donde el OCR de la página completa no leyó la tabla (encabezados grises): se vuelve a
    leer con OCR solo la franja de la tabla, entre '... así:' y 'CDP' / 'Total' / 'Objeto'.
    """
    rect = _franja(pagina, palabras, lambda n, w: w.rstrip().endswith("así:") or w.rstrip().endswith("asi:"),
                   lambda n, w: n in ("CDP", "TOTAL", "OBJETO"))
    if rect is None:
        return []
    return _filas_por_coordenadas(_palabras_recorte(pagina, rect), pagina)


def _certificado_bogdata(pagina, texto, tipo, palabras_ocr=None):
    """Formato BogData (UNIDAD EJECUTORA 01): tabla RUBRO / DESCRIPCIÓN / CONCEPTO / FONDO / VALOR."""
    m = re.search(r"N[uú]mero\s*[:.;,]?\s*(\d+)", texto)      # con OCR los ":" a veces salen como "."
    if not m:
        return []
    numero = m.group(1)
    cdp = re.search(r"\bCDP\s*[:.;,]?\s*(\d+)", texto)
    bogdata = re.search(r"Impresi[oó]n\s*[:.;,]?\s*\S+\s+\S+\s+(\d{6,})\s+\d{4}", texto)
    fecha = _fecha(texto)
    filas = []
    if palabras_ocr is not None:                 # página leída con OCR
        filas = (_filas_por_celdas(pagina, palabras_ocr) or _filas_por_coordenadas(palabras_ocr)
                 or _filas_por_franja(pagina, palabras_ocr))
    for tabla in (pagina.find_tables().tables if palabras_ocr is None else []):
        datos = tabla.extract()
        if not datos:
            continue
        encabezado = [_sin_tildes(_limpiar_celda(c)) for c in datos[0]]
        if "RUBRO" not in encabezado or "VALOR" not in encabezado:
            continue
        pos = {nombre: encabezado.index(nombre) for nombre in encabezado if nombre}
        for fila in datos[1:]:
            celdas = [c or "" for c in fila]
            rubro = re.sub(r"\s+", "", celdas[pos["RUBRO"]])
            if not rubro or any(_limpiar_celda(c).upper() == "TOTAL" for c in celdas):
                continue
            vig = celdas[pos["VIGENCIA FUTURA"]].strip() if "VIGENCIA FUTURA" in pos else ""
            filas.append({
                "Rubro": rubro,
                "Descripción Rubro": _limpiar_celda(celdas[pos.get("DESCRIPCION DEL RUBRO", -1)]) if "DESCRIPCION DEL RUBRO" in pos else None,
                "Concepto Gasto": _limpiar_celda(celdas[pos["CONCEPTO DEL GASTO"]]) if "CONCEPTO DEL GASTO" in pos else None,
                "Fondo": _limpiar_celda(celdas[pos["FONDO"]]) if "FONDO" in pos else None,
                "Valor CRP": _a_numero(celdas[pos["VALOR"]]),
                "_vigencia": vig,
            })
    if not filas:                       # la tabla no se detectó: se lee del texto
        filas = _filas_desde_texto(texto)
    salida = []
    for f in filas:
        num = numero + (f" (VF {f['_vigencia']})" if f["_vigencia"] else "")
        salida.append({
            "tipo": tipo,
            "NumeroCRP": num if tipo == "CRP" else None,
            "NumeroCDP": (cdp.group(1) if cdp else None) if tipo == "CRP" else num,
            "Rubro": f["Rubro"], "Descripción Rubro": f["Descripción Rubro"],
            "Concepto Gasto": f["Concepto Gasto"], "Fondo": f["Fondo"],
            "Valor CRP": f["Valor CRP"], "Fecha CRP": fecha,
            "CRP Interno BogData": bogdata.group(1) if bogdata else None,
        })
    return salida


def _certificado_fondo_cuenta(texto, tipo, pagina=None, palabras_ocr=None):
    """Formato SEVEN del Fondo Cuenta (UNIDAD EJECUTORA 2): 'Número: CRP - FC - 456', depósitos y valores."""
    m = re.search(r"N[uú]mero\s*[:.;,]?\s*(CRP|CDP)\s*-\s*FC\s*-\s*(\d+)", texto, re.I)
    if not m:
        return []
    numero = f"FC-{m.group(2)}"
    total = re.search(r"TOTAL\s*\n?\s*([\d,]+\.\d{2})", texto)
    antes_total = texto[: total.start()] if total else texto
    depositos = re.findall(r"(?m)^\s*(D\d{5,})\s*-\s*(.+?)\s*$", antes_total)
    valores = [v for v in re.findall(r"(?m)^\s*([\d,]+\.\d{2})\s*$", antes_total)]
    if not depositos and palabras_ocr is not None:
        # Escaneado: el OCR de la página completa no leyó la fila del depósito -> OCR solo de esa franja
        # (solo la parte izquierda, donde está el texto del depósito; la columna VALOR ya se leyó)
        rect = _franja(pagina, palabras_ocr, lambda n, w: n == "DEPOSITO", lambda n, w: n in ("CDP", "TOTAL"), 120)
        if rect is not None:
            rect.x1 = rect.x0 + rect.width * 0.58
            franja = ocr_recorte(pagina, rect)
            depositos = re.findall(r"(D\d{5,})\s*[-–]?\s*(.+?)(?=\s+D\d{5,}|\s+TOTAL|$)", franja)
            depositos = [(c, re.sub(r"[\"”“|]", "", d).strip()) for c, d in depositos]
            valores = valores or re.findall(r"(?m)\d{1,3}(?:,\d{3})+\.\d{2}", texto)
        if not depositos and valores:
            depositos = [(None, "")]
    cdps = re.findall(r"CDP Afectado:\s*(\d+)", texto)
    fecha = _fecha(texto)
    salida = []
    for i, (codigo, descripcion) in enumerate(depositos):
        salida.append({
            "tipo": tipo,
            "NumeroCRP": numero if tipo == "CRP" else None,
            "NumeroCDP": ", ".join(cdps) if tipo == "CRP" else numero,
            "Rubro": codigo, "Descripción Rubro": descripcion.strip() or None,
            "Concepto Gasto": None, "Fondo": "Fondo Cuenta",
            "Valor CRP": _a_numero(valores[i]) if i < len(valores) else None,
            "Fecha CRP": fecha, "CRP Interno BogData": None,
        })
    return salida


def _certificado_fondo_cuenta_antiguo(texto, tipo):
    """
    Formato anterior del Fondo Cuenta (2024-2025): 'Número: 502', 'DEPOSITO D2120016', 'VALOR 198.220.000',
    'CDP AFECTADO: 353', 'Se expide en Bogota D.C, a los 31 de enero de 2025'.
    """
    m = re.search(r"N[uú]mero\s*[:.;,]?\s*(\d+)", texto)
    if not m:
        return []
    numero = f"FC-{m.group(1)}"
    valor = re.search(r"VALOR\s*\n\s*(\d{1,3}(?:[.,]\d{3})+(?:[.,]\d{2})?)", texto)
    cdp = re.search(r"CDP\s+AFECTADO\s*:?\s*(\d+)", texto, re.I)
    deposito = re.search(r"DEPOSITO\s*\n\s*(.+)", texto)
    codigo = re.search(r"\b(D\d{5,})\b", texto)
    descripcion = None
    if not codigo and deposito and not re.match(r"\d{3} [A-Z]", deposito.group(1)):
        descripcion = deposito.group(1).strip()          # "2. CV ATENEA-FFDS VACUNAS"
    return [{
        "tipo": tipo,
        "NumeroCRP": numero if tipo == "CRP" else None,
        "NumeroCDP": (cdp.group(1) if cdp else None) if tipo == "CRP" else numero,
        "Rubro": codigo.group(1) if codigo else None, "Descripción Rubro": descripcion,
        "Concepto Gasto": None, "Fondo": "Fondo Cuenta",
        "Valor CRP": _a_numero(valor.group(1)) if valor else None,
        "Fecha CRP": _fecha(texto), "CRP Interno BogData": None,
    }]


def _tipo_certificado(texto, ocr=False):
    if not ocr:
        if "CERTIFICADO DE REGISTRO PRESUPUESTAL" in texto:
            return "CRP"
        if "CERTIFICADO DE DISPONIBILIDAD PRESUPUESTAL" in texto:
            return "CDP"
        return None
    n = re.sub(r"\s+", " ", _sin_tildes(texto))     # con OCR se tolera alguna diferencia de espacios
    if "CERTIFICADO" in n and "REGISTRO PRESUPUESTAL" in n:
        return "CRP"
    if "CERTIFICADO" in n and "DISPONIBILIDAD PRESUPUESTAL" in n:
        return "CDP"
    return None


def _leer_certificado(pagina, texto, tipo, palabras_ocr=None):
    if re.search(r"N[uú]mero\s*[:.;,]?\s*(CRP|CDP)\s*-\s*FC", texto, re.I):
        return _certificado_fondo_cuenta(texto, tipo, pagina, palabras_ocr)
    if re.search(r"FONDO CUENTA", texto, re.I) and re.search(r"CDP\s+AFECTADO", texto, re.I) \
            and not re.search(r"\bRUBRO\b", texto):
        return _certificado_fondo_cuenta_antiguo(texto, tipo)
    return _certificado_bogdata(pagina, texto, tipo, palabras_ocr)


def certificados_de_pdf(datos):
    """
    Revisa TODAS las páginas y devuelve los CRP y CDP que encuentre (una entrada por rubro).
    1) Páginas con texto normal.  2) Si no encontró nada, páginas escaneadas o ilegibles con OCR.
    """
    doc = pymupdf.open(stream=datos, filetype="pdf")
    salida, por_ocr = [], []
    for k, pagina in enumerate(doc):
        texto = _arreglar_fuente(pagina.get_text())
        if texto_ilegible(texto):
            por_ocr.append(k)
            continue
        tipo = _tipo_certificado(texto)
        if tipo:
            salida.extend(_leer_certificado(pagina, texto, tipo))
    if not salida and USAR_OCR and RUTA_TESSDATA:
        for k in por_ocr[:MAX_PAGINAS_OCR]:
            pagina = doc[k]
            texto, palabras = pagina_ocr(pagina)
            tipo = _tipo_certificado(texto or "", ocr=True)
            if tipo:
                salida.extend(_leer_certificado(pagina, texto, tipo, palabras))
    doc.close()
    return salida


def _valor_cop(lineas):
    """Primer valor en pesos dentro de las líneas: '1.200.000.000 COP' o '365.859.701' + 'COP'."""
    for k, l in enumerate(lineas):
        if re.fullmatch(r"[\d.]+(,\d+)?\s*COP", l):
            return _a_numero(l.replace("COP", ""))
        if re.fullmatch(r"[\d.]{5,}(,\d+)?", l) and k + 1 < len(lineas) and lineas[k + 1] == "COP":
            return _a_numero(l)
    return None


def secop_en_ejecucion(datos):
    """
    Página de SECOP 'En ejecución'.
    - 'CDP/Vigencias Futuras': códigos de tipo CDP y su 'Valor a utilizar'.
    - 'Compromiso presupuestal de gastos': cada compromiso (= CRP) con su 'Valor compromiso'.
    Devuelve un registro por compromiso.
    """
    doc = pymupdf.open(stream=datos, filetype="pdf")
    texto = "\n".join(p.get_text() for p in doc)
    doc.close()
    lineas = [l.strip() for l in texto.splitlines()]

    # CDP: tabla "CDP/Vigencias Futuras" y tabla de "Sistema de Presupuesto y Giro de Regalías"
    cdps, valores_cdp = [], []
    inicios = [x for x in (texto.find("Sistema de Presupuesto y Giro"), texto.find("CDP/Vigencias Futuras")) if x != -1]
    if inicios:
        fin = texto.find("Compromiso presupuestal de gastos", min(inicios))
        bloque = [l.strip() for l in texto[min(inicios): fin if fin != -1 else None].splitlines()]
        for k in range(1, len(bloque)):
            if bloque[k] != "CDP":
                continue
            codigo = next((bloque[k - d] for d in (1, 2) if k - d >= 0 and re.fullmatch(r"\d{1,7}", bloque[k - d])), None)
            if codigo:
                cdps.append(codigo)
                vals = [_valor_cop([l]) for l in bloque[k + 1: k + 6]]
                vals = [v for v in vals if v]
                valores_cdp.append(vals[-1] if vals else None)     # último = "Valor a utilizar"
    cdps = list(dict.fromkeys(cdps))

    # Compromisos (CRP): "Presupuestal (CDP)", "Regalías (CDP)", "Vigencia Futura (AVF)"...
    compromisos = []
    for k in range(len(lineas) - 2):
        if not re.fullmatch(r"\d+", lineas[k]):
            continue
        siguiente = " ".join(lineas[k + 1: k + 3])
        m = re.match(r"^[A-Za-zÁÉÍÓÚáéíóúñ ]{3,30}\((CDP|AVF)\)", siguiente)
        if not m:
            continue
        etiqueta = lineas[k] + (" (AVF)" if m.group(1) == "AVF" else "")
        compromisos.append((etiqueta, _valor_cop(lineas[k + 3: k + 12])))
    compromisos = list(dict.fromkeys(compromisos))

    if not cdps and not compromisos:
        return []
    valor_cdp = next((v for v in valores_cdp if v), None)   # 'Valor a utilizar' del primer CDP con valor
    base = {"tipo": "En ejecución", "NumeroCDP": ", ".join(cdps) or None,
            "Rubro": None, "Descripción Rubro": None, "Concepto Gasto": None, "Fondo": None,
            "Fecha CRP": None, "CRP Interno BogData": None}
    if not compromisos:
        return [dict(base, NumeroCRP=None, **{"Valor CRP": valor_cdp})]
    return [dict(base, NumeroCRP=codigo, **{"Valor CRP": valor if valor is not None else valor_cdp})
            for codigo, valor in compromisos]


def certificados_de_archivo(nombre, datos, fuente):
    """
    Devuelve (lista de registros, estado) para un archivo descargado (PDF o comprimido).
    fuente: 'CRP', 'CuentaCobro1', 'CDP' o 'En ejecución'.
    """
    ext = Path(nombre).suffix.lower()
    if ext == ".pdf":
        pdfs = [(nombre, datos)]
    elif ext in EXT_COMPRIMIDOS:
        pdfs = [(n, b) for n, b in _archivos_de_comprimido(nombre, datos) if n.lower().endswith(".pdf")]
    else:
        return [], f"extensión {ext} no soportada"
    registros, cdps = [], []
    for n, b in pdfs:
        try:
            if fuente == "En ejecución":
                registros += secop_en_ejecucion(b)
            else:
                encontrados = certificados_de_pdf(b)
                quiero = "CDP" if fuente == "CDP" else "CRP"
                registros += [r for r in encontrados if r["tipo"] == quiero]
                cdps += [dict(r, _solo_cdp=True) for r in encontrados if r["tipo"] == "CDP"]
        except Exception:
            continue
    if not registros and cdps and fuente != "CDP":
        # El documento se llama CRP pero en realidad es un certificado de DISPONIBILIDAD (CDP):
        # se guarda como respaldo (sin número de CRP) por si ninguna otra fuente trae el CRP
        return cdps, "solo CDP (el documento es un certificado de disponibilidad)"
    return registros, ("ok" if registros else "no se encontró el certificado")

### **2. Extracción por contrato**

In [ ]:
# ------------------------------------------------------------------
# 7.2 Extracción por contrato (CRP directo -> Cuenta de cobro 1 -> CDP -> En ejecución)
# ------------------------------------------------------------------
# Reutiliza la configuración, la descarga y la caché de la sección 6.

# Al mejorar las reglas se sube este número: los documentos ya leídos se vuelven a leer
# (desde los archivos guardados, sin descargarlos de nuevo).
VERSION_REGLAS_CRP = 5
# Desde esta versión, los documentos que YA tienen datos de CRP no se vuelven a leer (solo los vacíos)
CONSERVAR_CRP_CON_RESULTADO_DESDE = 4

VALOR_CRP_COMO_TOTAL = False   # True = "Valor CRP" lleva la SUMA de todos los CRP del contrato (no la lista)

archivo_resultados_crp = Path(CARPETA_CACHE) / "resultados_crp.json"
resultados_crp = (json.loads(archivo_resultados_crp.read_text(encoding="utf-8"))
                  if archivo_resultados_crp.exists() else {})

COLUMNAS_CRP = ["NumeroCRP", "NumeroCDP", "Rubro", "Descripción Rubro", "Concepto Gasto",
                "Fondo", "Valor CRP", "Fecha CRP", "CRP Interno BogData"]

# (nombre de la fuente, columna que la identifica). El orden es la prioridad.
FUENTES_CRP = [
    ("CRP", "CRP/CDP"),
    ("CuentaCobro1", "CuentaCobro1"),
    ("CDP", "CDP Exc."),
    ("En ejecución", "contratoEjecucion"),
]


def guardar_resultados_crp():
    archivo_resultados_crp.write_text(json.dumps(resultados_crp, ensure_ascii=False), encoding="utf-8")


def procesar_crp(filas, fuente):
    """Descarga (con caché) y lee los documentos de una fuente."""
    clave = lambda f: f"{fuente}|{f['ID Documento']}"
    def hay_que_leer(f):
        r = resultados_crp.get(clave(f), {})
        if r.get("v") == VERSION_REGLAS_CRP:
            return False
        return not (r.get("registros") and r.get("v", 0) >= CONSERVAR_CRP_CON_RESULTADO_DESDE)
    pendientes = [f for _, f in filas.iterrows() if hay_que_leer(f)]
    print(f"Fuente {fuente}: {len(filas)} documentos ({len(filas) - len(pendientes)} ya procesados antes)")
    if not pendientes:
        return
    with ThreadPoolExecutor(MAX_DESCARGAS_SIMULTANEAS) as ex:
        futuros = {ex.submit(descargar, f): str(f["ID Documento"]) for f in pendientes}
        for k, fut in enumerate(as_completed(futuros), 1):
            ok, error = fut.result()
            if not ok:
                fallos_descarga[futuros[fut]] = error
            if k % 50 == 0 or k == len(pendientes):
                print(f"   descargados {k}/{len(pendientes)}")
    for k, f in enumerate(pendientes, 1):
        if str(f["ID Documento"]) in fallos_descarga:
            continue
        ruta = ruta_local(f)
        try:
            registros, estado = certificados_de_archivo(ruta.name, ruta.read_bytes(), fuente)
        except Exception as e:
            registros, estado = [], f"error al leer ({type(e).__name__})"
        resultados_crp[clave(f)] = {"registros": registros, "estado": estado, "v": VERSION_REGLAS_CRP}
        if k % 50 == 0 or k == len(pendientes):
            guardar_resultados_crp()
            print(f"   leídos {k}/{len(pendientes)}")
    guardar_resultados_crp()


def _orden_fecha(registro):
    """Orden: fecha (sin fecha al final) y luego número del CRP/CDP."""
    numero = re.search(r"\d+", str(registro.get("NumeroCRP") or registro.get("NumeroCDP") or ""))
    numero = int(numero.group()) if numero else 0
    try:
        d, m, a = registro["Fecha CRP"].split("/")
        return (0, int(a), int(m), int(d), numero)
    except Exception:
        return (1, 0, 0, 0, numero)


con_contrato = docs["Número de Contrato"].notna()
crp_contrato = {}     # contrato -> (índice de la fila, registros, fuente)
solo_cdp = {}         # respaldo: documentos "CRP" que en realidad son un CDP (se usan si no aparece el CRP)
log_crp = []

for fuente, columna in FUENTES_CRP:
    filas = docs[con_contrato & (docs[columna] == "Si")
                     & ~docs["Número de Contrato"].isin(list(crp_contrato))].copy()
    if filas.empty:
        continue
    procesar_crp(filas, fuente)

    filas["_fecha"] = pd.to_datetime(filas["Fecha Carga"], errors="coerce")
    filas["_id"] = pd.to_numeric(filas["ID Documento"], errors="coerce")
    filas = filas.sort_values(["_fecha", "_id"], ascending=False)    # más reciente primero

    for contrato, grupo in filas.groupby("Número de Contrato", sort=False):
        registros, vistos, fila_destino = [], set(), None
        for idx, f in grupo.iterrows():
            r = resultados_crp.get(f"{fuente}|{f['ID Documento']}")
            if r is None:
                r = {"registros": [], "estado": f"descarga falló ({fallos_descarga.get(str(f['ID Documento']), '')})"}
            nuevos = 0
            for reg in r["registros"]:
                llave = (reg["NumeroCRP"], reg["NumeroCDP"], reg["Rubro"], reg["Valor CRP"])
                if llave not in vistos:            # el mismo CRP subido dos veces cuenta una vez
                    vistos.add(llave)
                    registros.append(reg)
                    nuevos += 1
            if r["registros"] and fila_destino is None:
                fila_destino = idx                   # fila del documento más reciente con datos
            log_crp.append({"Número de Contrato": contrato,
                            "Referencia del contrato": f.get("Referencia del contrato"),
                            "Año": f.get("Año"),
                            "ID Documento": f["ID Documento"],
                            "Nombre Documento": f["Nombre Documento"], "Fuente": fuente,
                            "Estado": r["estado"], "Registros encontrados": len(r["registros"]),
                            "Registros nuevos": nuevos,
                            "URL Descarga Documento": f["URL Descarga Documento"]})
            if fuente == "En ejecución" and registros:
                break                                # basta con el reporte de SECOP más reciente
        if registros:
            registros.sort(key=_orden_fecha)         # del más antiguo al más reciente
            if all(reg.get("_solo_cdp") for reg in registros):
                solo_cdp.setdefault(contrato, (fila_destino, registros, f"{fuente} (solo CDP)"))
            else:
                crp_contrato[contrato] = (fila_destino, [reg for reg in registros if not reg.get("_solo_cdp")], fuente)

# Contratos sin ningún CRP: se usa el CDP encontrado en un documento llamado CRP / cuenta de cobro
for contrato, datos in solo_cdp.items():
    if contrato not in crp_contrato:
        crp_contrato[contrato] = datos

# Valores finales por contrato (se agregan a la tabla contractual en la sección 8)
datos_crp = {}


def _texto(v):
    return "-" if v is None or v == "" else str(v)


for contrato, (idx, registros, fuente) in crp_contrato.items():
    fila = {}
    for col in COLUMNAS_CRP:
        valores = [r.get(col) for r in registros]
        if col == "Valor CRP" and VALOR_CRP_COMO_TOTAL:
            fila[col] = sum(v for v in valores if v)
        elif len(valores) == 1:
            fila[col] = valores[0]
        elif any(v not in (None, "") for v in valores):
            fila[col] = " | ".join(_texto(v) for v in valores)
        else:
            fila[col] = None
    datos_crp[contrato] = fila

log_crp = pd.DataFrame(log_crp)
for contrato, (idx, registros, fuente) in crp_contrato.items():
    log_crp.loc[log_crp["Número de Contrato"] == contrato, "Fuente usada"] = fuente

# ------------------------------------------------------------------
# Resumen
# ------------------------------------------------------------------
print(f"\nContratos con datos de CRP: {len(crp_contrato)}")
print(pd.Series([fu for _, _, fu in crp_contrato.values()], dtype="object")
      .value_counts().rename("Fuente usada").to_string())
if not log_crp.empty:
    print("\nEstado de los documentos leídos:")
    print(log_crp["Estado"].value_counts().to_string())

### **3. Revisión de resultados**

In [ ]:
# ------------------------------------------------------------------
# 7.3 Revisión
# ------------------------------------------------------------------
ejemplo = pd.DataFrame.from_dict(datos_crp, orient="index", columns=COLUMNAS_CRP).head(5)
display(ejemplo)

# Contratos que tenían documentos candidatos pero no se pudo extraer nada
if not log_crp.empty:
    sin_crp = (log_crp[~log_crp["Número de Contrato"].isin(list(crp_contrato))]
               .groupby("Número de Contrato")
               .agg(documentos=("Nombre Documento", " | ".join), estados=("Estado", " | ".join))
               .reset_index())
    print(f"Contratos sin datos de CRP: {len(sin_crp)}")
    display(sin_crp.head(20))

## **8. Columnas documentales en la tabla contractual**

A la tabla contractual (tal como viene, una fila por contrato) se le agregan:

+ URLs de los documentos clasificados: `crp_cdp` (CRP/CDP o CDP Exc.), `estudios_previos`, `minutas`, `contrato_ejecucion`. Si hay varios documentos se separan con ` | ` (del más antiguo al más reciente).
+ Lo extraído de los documentos: `obligaciones_específicas`, `numero_crp`, `numero_cdp`, `rubro`, `descripción_rubro`, `concepto_gasto`, `fondo`, `valor_crp`, `fecha_crp`, `crp_interno_bogdata`.


In [ ]:
# ------------------------------------------------------------------
# 8. Columnas documentales en la tabla contractual (una fila por contrato)
# ------------------------------------------------------------------
SEPARADOR_URL = " | "

COLUMNAS_DOCUMENTALES = ["crp_cdp", "estudios_previos", "minutas", "contrato_ejecucion",
                         "obligaciones_específicas", "numero_crp", "numero_cdp", "rubro",
                         "descripción_rubro", "concepto_gasto", "fondo", "valor_crp", "fecha_crp",
                         "crp_interno_bogdata"]
# Si la celda se ejecuta de nuevo, se parte otra vez de la tabla tal como vino
tabla_contractual = tabla_contractual.drop(columns=[c for c in COLUMNAS_DOCUMENTALES if c in tabla_contractual.columns])

orden_docs = docs[docs["Número de Contrato"].notna()].assign(
    _fecha=pd.to_datetime(docs["Fecha Carga"], errors="coerce"),
    _id=pd.to_numeric(docs["ID Documento"], errors="coerce"),
).sort_values(["Número de Contrato", "_fecha", "_id"])              # del más antiguo al más reciente


def _urls(columnas):
    """contrato -> 'url1 | url2 | ...' (sin repetir) de los documentos con 'Si' en alguna de las columnas."""
    marca = orden_docs[columnas].eq("Si").any(axis=1)
    sel = orden_docs[marca & orden_docs["URL Descarga Documento"].notna()]
    return sel.groupby("Número de Contrato")["URL Descarga Documento"].agg(
        lambda s: SEPARADOR_URL.join(dict.fromkeys(str(u).strip() for u in s)))


# URLs de los documentos clasificados
tabla_contractual["crp_cdp"] = llave_contrato.map(_urls(["CRP/CDP", "CDP Exc."]))
tabla_contractual["estudios_previos"] = llave_contrato.map(_urls(["Estudios/documentos previos"]))
tabla_contractual["minutas"] = llave_contrato.map(_urls(["Minutas"]))
tabla_contractual["contrato_ejecucion"] = llave_contrato.map(_urls(["contratoEjecucion"]))

# Obligaciones específicas (sección 6)
tabla_contractual["obligaciones_específicas"] = llave_contrato.map(obligaciones_texto)

# Datos del CRP (sección 7)
nombres_crp = {
    "NumeroCRP": "numero_crp",
    "NumeroCDP": "numero_cdp",
    "Rubro": "rubro",
    "Descripción Rubro": "descripción_rubro",
    "Concepto Gasto": "concepto_gasto",
    "Fondo": "fondo",
    "Valor CRP": "valor_crp",
    "Fecha CRP": "fecha_crp",
    "CRP Interno BogData": "crp_interno_bogdata",
}
for original, nueva in nombres_crp.items():
    tabla_contractual[nueva] = llave_contrato.map({c: d.get(original) for c, d in datos_crp.items()})

# Límite de caracteres de una celda de Excel
for col in ["crp_cdp", "estudios_previos", "minutas", "contrato_ejecucion", "obligaciones_específicas"]:
    largo = tabla_contractual[col].astype("string").str.len()
    largos = largo > 32000
    if largos.any():
        tabla_contractual.loc[largos, col] = tabla_contractual.loc[largos, col].str[:32000] + " …[recortado]"

# ------------------------------------------------------------------
# Resumen
# ------------------------------------------------------------------
print(f"Tabla contractual: {len(tabla_contractual)} filas, {tabla_contractual.shape[1]} columnas")
for col in COLUMNAS_DOCUMENTALES:
    print(f"   {col:26}: {tabla_contractual[col].notna().sum()} contratos con dato")


## **9. Exportar a Excel**

Se genera `tabla_contractual_atenea.xlsx`: la hoja **Tabla contractual** (una fila por contrato) y, si `EXPORTAR_LOGS = True`, las hojas **Log obligaciones** y **Log CRP** para revisar qué se leyó de cada documento.

In [ ]:
ruta = CARPETA_SALIDA      # se define en la configuración general, al inicio del notebook
os.makedirs(ruta, exist_ok=True)

EXPORTAR_LOGS = True     # False = solo la hoja de la tabla contractual

archivo_contractual = os.path.join(ruta, "tabla_contractual_atenea.xlsx")
with pd.ExcelWriter(
    archivo_contractual, engine="openpyxl",
    date_format="DD/MM/YYYY", datetime_format="DD/MM/YYYY",
) as writer:
    tabla_contractual.to_excel(writer, index=False, sheet_name="Tabla contractual")
    if EXPORTAR_LOGS and "log_obligaciones" in globals():
        log_obligaciones.to_excel(writer, index=False, sheet_name="Log obligaciones")
    if EXPORTAR_LOGS and "log_crp" in globals():
        log_crp.to_excel(writer, index=False, sheet_name="Log CRP")

print(f"Archivo exportado: {os.path.abspath(archivo_contractual)} ({len(tabla_contractual)} contratos)")
